In [1]:
# ============================================================
# 019_pdf_fetch_and_ingest_to_drive_notion
# ============================================================
#
# Overview
# ----------------
# This notebook automates the retrieval of open-access (free-to-download) PDFs
# for a curated and prioritized list of academic papers, and ingests successfully
# retrieved PDFs into Google Drive and a Notion literature database.
#
# It is designed to be executed as a daily or on-demand batch within a
# research-agent workflow, where candidate papers have already been discovered,
# scored, and filtered by an upstream discovery process
# (e.g., 018_seed_corpus_2023plus_discovery.ipynb or 020_backward_fill_high_citation_core.ipynb).
#
# By default, the notebook operates on a *pre-filtered candidate set* where
# open-access likelihood has already been assessed upstream.
# As a result, this notebook typically attempts PDF downloads only for a
# subset of candidates (e.g., OA-likely papers), not necessarily the full
# upstream candidate list.
#
# The notebook emphasizes robustness and operational usability:
# - multiple ordered PDF URL candidates per paper
# - retry, fallback, and failure classification logic
# - safe credential and token handling
# - idempotent ingestion into Google Drive and Notion
# - clear separation between automated steps and human review
#
#
# Inputs / Outputs
# ----------------
# Inputs:
# - Candidate paper list (CSV, produced by an upstream discovery / backfill step)
#   Typical fields include:
#     - canonical identifiers: doi, openalex_id (preferred), or paper_id
#     - title, publication_year, venue
#     - cited_by_count
#     - rq_score / priority_score
#     - oa_status, oa_url
#     - candidate PDF URLs (explicit or inferred)
#
# Outputs:
# - PDF fetch results (per attempted paper):
#     - success flag (ok)
#     - chosen / final URL
#     - local file path, file size, hash
#     - failure code and message (if failed)
#
# - Google Drive ingestion results:
#     - Drive file ID
#     - web-accessible Drive link
#     - deduplication outcome
#
# - Notion database records:
#     - created or updated page ID
#     - structured literature fields (Core Idea, Methods, Findings, etc.)
#     - linked Drive PDF (if available)
#
# - Failure logs (for human follow-up):
#     - identifiers (doi / openalex_id)
#     - failure stage (fetch / drive / notion)
#     - failure reason and diagnostics
#
#
# Structure
# ----------------
# Cell 00: Notebook purpose and execution assumptions
# Cell 01: Imports and retry / HTTP utilities
# Cell 02: Secure credential loading (Google Drive / Notion)
# Cell 03: API client initialization and connectivity checks
# Cell 04: Load candidate papers and select processing targets
#          (idempotent filtering; skip already-processed papers)
# Cell 05: Construct ordered PDF URL candidates for each paper
#          (OA URLs, publisher patterns, heuristics)
# Cell 06: PDF download function
#          (streaming, validation, hashing, size checks)
# Cell 07: Retry and fallback strategy for failed downloads
# Cell 08: Optional alternative PDF discovery (private add-on hook)
# Cell 09: Google Drive upload and duplicate handling
# Cell 10: Notion upsert logic
#          (PDF-based text extraction + LLM-assisted structuring)
# Cell 11: Main processing loop (fetch → drive → notion)
# Cell 12: Aggregation of success / failure metrics
# Cell 13: Persist results and logs for downstream analysis
# Cell 14: Human-review summary
#          (high-priority failures; manual acquisition candidates)
#
#
# Notes
# ----------------
# - Secrets and tokens must NOT be hard-coded in this notebook.
#   Use environment variables or external secret files excluded from version control.
# - The notebook is designed to be restartable and idempotent:
#   previously processed papers should be safely skipped unless explicitly re-run.
# - Most steps are fully automated; human involvement is intentionally concentrated
#   in reviewing high-priority failures, deciding manual acquisition, and refining
#   upstream discovery heuristics.
# - This notebook is intended to operate as part of a broader research OS,
#   paired with:
#     - a weekly corpus discovery / backfill pipeline (018 / 020)
#     - a daily human-in-the-loop research review cycle
#
# ============================================================


In [8]:
# ============================================================
# Cell 01: Imports and retry / HTTP utilities
# ============================================================
#
# This cell defines all core imports and shared HTTP / retry utilities
# used throughout the notebook.
#
# Design principles:
# - Explicit imports for reproducibility
# - Centralized retry / backoff logic
# - Clear failure signaling (no silent exceptions)
# - Safe defaults for large PDF downloads (streaming)
#

# --- Standard library ---
import os
import time
import json
import hashlib
import mimetypes
from typing import List, Dict, Optional, Tuple

# --- Third-party libraries ---
import requests
import pandas as pd
from tqdm import tqdm

# Retry / backoff
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
)

# HTTP exceptions
from requests.exceptions import (
    RequestException,
    Timeout,
    ConnectionError,
    HTTPError,
)

# ============================================================
# HTTP configuration
# ============================================================

# User-Agent is critical to avoid being blocked by publishers
DEFAULT_HEADERS = {
    "User-Agent": (
        "ResearchAgent/1.0 "
        "(PDF ingestion for academic research; contact: internal)"
    ),
    "Accept": "application/pdf,application/octet-stream;q=0.9,*/*;q=0.8",
}

# Timeouts: (connect timeout, read timeout)
DEFAULT_TIMEOUT = (10, 60)

# Chunk size for streaming downloads (bytes)
STREAM_CHUNK_SIZE = 1024 * 1024  # 1MB


# ============================================================
# Retry policy
# ============================================================

# Retry only for network / transient errors
RETRYABLE_EXCEPTIONS = (
    Timeout,
    ConnectionError,
    HTTPError,
    RequestException,
)

@retry(
    reraise=True,
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=2, min=2, max=20),
    retry=retry_if_exception_type(RETRYABLE_EXCEPTIONS),
)
def http_get_with_retry(
    url: str,
    headers: Optional[Dict[str, str]] = None,
    stream: bool = False,
    timeout: Tuple[int, int] = DEFAULT_TIMEOUT,
) -> requests.Response:
    """
    Perform an HTTP GET request with retry and exponential backoff.

    Parameters
    ----------
    url : str
        Target URL.
    headers : dict, optional
        HTTP headers (merged with DEFAULT_HEADERS).
    stream : bool
        Whether to stream the response (required for large PDFs).
    timeout : tuple
        (connect timeout, read timeout).

    Returns
    -------
    requests.Response
        Successful HTTP response.

    Raises
    ------
    requests.exceptions.RequestException
        If all retries fail.
    """
    merged_headers = DEFAULT_HEADERS.copy()
    if headers:
        merged_headers.update(headers)

    response = requests.get(
        url,
        headers=merged_headers,
        timeout=timeout,
        stream=stream,
        allow_redirects=True,
    )

    # Raise for HTTP errors (4xx / 5xx)
    response.raise_for_status()
    return response


# ============================================================
# Helper utilities
# ============================================================

def is_pdf_response(response: requests.Response) -> bool:
    """
    Heuristically determine whether an HTTP response looks like a PDF.

    Checks Content-Type and (optionally) the first bytes of the body.
    """
    content_type = response.headers.get("Content-Type", "").lower()
    if "pdf" in content_type:
        return True
    return False


def compute_file_hash(file_path: str, algo: str = "sha256") -> str:
    """
    Compute a hash of a local file to detect duplicates.

    Parameters
    ----------
    file_path : str
        Path to the file.
    algo : str
        Hash algorithm (default: sha256).

    Returns
    -------
    str
        Hex digest of the file.
    """
    h = hashlib.new(algo)
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def safe_filename(text: str, max_length: int = 120) -> str:
    """
    Generate a filesystem-safe filename from text.

    Removes problematic characters and truncates length.
    """
    keepchars = (" ", ".", "_", "-")
    cleaned = "".join(c for c in text if c.isalnum() or c in keepchars)
    cleaned = cleaned.strip().replace(" ", "_")
    return cleaned[:max_length]


# ============================================================
# Failure reason codes (shared vocabulary)
# ============================================================

FAILURE_CODES = {
    "http_error": "HTTP error (4xx / 5xx)",
    "timeout": "Network timeout",
    "not_pdf": "Response is not a PDF",
    "too_small": "Downloaded file is suspiciously small",
    "hash_duplicate": "Duplicate file detected by hash",
    "unknown": "Unknown error",
}

# End of Cell 01


In [17]:
# ============================================================
# Cell 02: Secure credential loading (Google Drive / Notion)
# ============================================================
#
# This cell loads secrets and configuration safely from env.txt and validates
# that the required credentials for Notion and Google Drive are available.
#
# Principles:
# - Never hard-code tokens in the notebook
# - Fail fast if required secrets are missing
# - Perform safe connectivity checks (no secret exposure)
# - Support flexible local paths via env.txt
#

import pathlib
from dotenv import load_dotenv

# ------------------------------------------------------------
# Helper: require_env
# ------------------------------------------------------------
def require_env(key: str) -> str:
    """
    Read an environment variable and raise a clear error if missing.
    """
    val = os.getenv(key)
    if val is None or str(val).strip() == "":
        raise ValueError(
            f"{key} could not be loaded from env.txt (or environment). "
            f"Please set {key} in env.txt."
        )
    return val


# ------------------------------------------------------------
# Load environment variables
# ------------------------------------------------------------
# Explicitly load env.txt (instead of default .env)
load_dotenv("env.txt")
print("🔧 Environment variables loaded from env.txt")


# ============================================================
# Notion configuration
# ============================================================
NOTION_TOKEN = require_env("NOTION_TOKEN")
NOTION_VERSION = require_env("NOTION_VERSION")  # e.g., "2022-06-28" or your pinned version

# Database IDs (set in env.txt)
# - NOTION_LIT_DB_ID: Literature / Papers database (recommended for 019)
# - NOTION_PAPERS_DB_ID: Alternative papers DB key (legacy in some notebooks)
# - NOTION_RQ_DB_ID: Research Question DB (fallback only)
NOTION_LIT_DB_ID = os.getenv("NOTION_LIT_DB_ID")
NOTION_PAPERS_DB_ID = os.getenv("NOTION_PAPERS_DB_ID")
NOTION_RQ_DB_ID = os.getenv("NOTION_RQ_DB_ID")  # legacy / optional

# Choose which DB ID to use (prefer NOTION_LIT_DB_ID for 019)
if NOTION_LIT_DB_ID and str(NOTION_LIT_DB_ID).strip():
    NOTION_DB_ID = NOTION_LIT_DB_ID
    print("✅ Using NOTION_LIT_DB_ID as NOTION_DB_ID (primary for 019).")
elif NOTION_PAPERS_DB_ID and str(NOTION_PAPERS_DB_ID).strip():
    NOTION_DB_ID = NOTION_PAPERS_DB_ID
    print("ℹ️ Using NOTION_PAPERS_DB_ID as NOTION_DB_ID (fallback). Consider setting NOTION_LIT_DB_ID.")
else:
    NOTION_DB_ID = require_env("NOTION_RQ_DB_ID")  # final fallback
    print("⚠️ Using NOTION_RQ_DB_ID as NOTION_DB_ID (final fallback). Consider setting NOTION_LIT_DB_ID.")

# Optional (only if you use it elsewhere)
NOTION_RQ_DATA_SOURCE_ID = os.getenv("NOTION_RQ_DATA_SOURCE_ID")

NOTION_HEADERS = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Notion-Version": NOTION_VERSION,
    "Content-Type": "application/json",
}

# Quick auth check (safe: does not reveal token)
try:
    r = requests.get("https://api.notion.com/v1/users/me", headers=NOTION_HEADERS, timeout=30)
    if r.status_code == 200:
        print("✅ Notion auth OK")
    else:
        print("⚠️ Notion auth check failed:", r.status_code, r.text[:200])
except Exception as e:
    print("⚠️ Notion auth check error:", type(e).__name__, str(e))



# ============================================================
# Google Drive configuration
# ============================================================

# NOTE:
# For uploading PDFs, you need a scope that allows file creation.
# If you keep drive.readonly, uploads will fail.
# Recommended for ingestion:
DRIVE_SCOPES = ["https://www.googleapis.com/auth/drive"] 

# If your environment requires full Drive access, use:
# DRIVE_SCOPES = ["https://www.googleapis.com/auth/drive"]

# ------------------------------------------------------------
# OAuth client secret handling
# ------------------------------------------------------------
# Priority:
# 1) GOOGLE_OAUTH_CLIENT_SECRET_JSON in env.txt (path)
# 2) Default local filename (checked in working directory)
#
DEFAULT_GOOGLE_CLIENT_SECRET = (
    "client_secret_750875982200-85rnsoqhr2af2b13peueev0bm60q22sh.apps.googleusercontent.com.json"
)

GOOGLE_OAUTH_CLIENT_SECRET_JSON = os.getenv("GOOGLE_OAUTH_CLIENT_SECRET_JSON")

if GOOGLE_OAUTH_CLIENT_SECRET_JSON:
    client_secret_path = pathlib.Path(GOOGLE_OAUTH_CLIENT_SECRET_JSON)
else:
    client_secret_path = pathlib.Path(DEFAULT_GOOGLE_CLIENT_SECRET)

if not client_secret_path.exists():
    raise ValueError(
        "Google OAuth client secret JSON not found.\n"
        "Either:\n"
        "  - set GOOGLE_OAUTH_CLIENT_SECRET_JSON in env.txt, or\n"
        f"  - place the file at: {DEFAULT_GOOGLE_CLIENT_SECRET}"
    )

print(f"✅ Google OAuth client secret located: {client_secret_path}")


# ------------------------------------------------------------
# Token cache path (configurable via env.txt)
# ------------------------------------------------------------
GOOGLE_TOKEN_JSON = os.getenv("GOOGLE_TOKEN_JSON", "google_token.json")
token_cache_path = pathlib.Path(GOOGLE_TOKEN_JSON)

# Destination Drive folder where PDFs will be saved
DRIVE_FOLDER_ID = require_env("DRIVE_FOLDER_ID")

print(f"📁 Drive target folder ID loaded (DRIVE_FOLDER_ID).")
print(f"🪪 Google token cache path: {token_cache_path}")


# ============================================================
# Optional: OpenAI key (only if used in later cells)
# ============================================================
# This notebook (019) may not strictly require OpenAI, but we keep this block
# if you reuse summarization/refinement in downstream steps.
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY:
    print("🔑 OPENAI_API_KEY loaded successfully (optional for 019)")
else:
    print("ℹ️ OPENAI_API_KEY not set (optional for 019).")


# ============================================================
# Safety notes
# ============================================================
# - Do NOT print tokens or client secret content.
# - Ensure env.txt and token cache files are excluded from version control (gitignore).
# - If running in CI or scheduled jobs, use environment variables / secret manager instead of env.txt.
#
# End of Cell 02


🔧 Environment variables loaded from env.txt
✅ Using NOTION_LIT_DB_ID as NOTION_DB_ID (primary for 019).
✅ Notion auth OK
✅ Google OAuth client secret located: client_secret_750875982200-85rnsoqhr2af2b13peueev0bm60q22sh.apps.googleusercontent.com.json
📁 Drive target folder ID loaded (DRIVE_FOLDER_ID).
🪪 Google token cache path: google_token.json
🔑 OPENAI_API_KEY loaded successfully (optional for 019)


In [10]:
# ============================================================
# Cell 03: API client initialization and connectivity checks
# ============================================================
#
# This cell initializes Google Drive and Notion access and performs
# safe connectivity checks.
#
# Key diagnostics:
# - Identify which Google account is authorized (Drive 'about.get')
# - Validate target folder ID access (files.get)
# - List a small sample of items in the folder (files.list)
#
# Common pitfalls:
# - drive.file scope may not "see" an existing folder created outside the app
# - wrong folder ID (file ID vs folder ID)
# - folder is in a Shared Drive but the user/app lacks access
#

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


# ------------------------------------------------------------
# Google Drive: credential bootstrap (token cache + refresh + OAuth flow)
# ------------------------------------------------------------
def get_drive_credentials(client_secret_json_path: str, token_json_path: str, scopes: List[str]) -> Credentials:
    creds = None
    token_path = pathlib.Path(token_json_path)

    if token_path.exists():
        creds = Credentials.from_authorized_user_file(str(token_path), scopes=scopes)

    if creds and creds.expired and creds.refresh_token:
        try:
            creds.refresh(Request())
        except Exception as e:
            print("⚠️ Token refresh failed:", type(e).__name__, str(e))
            creds = None

    # If still invalid, run OAuth flow (interactive)
    if not creds or not creds.valid:
        flow = InstalledAppFlow.from_client_secrets_file(str(client_secret_json_path), scopes=scopes)
        creds = flow.run_local_server(port=0)
        token_path.write_text(creds.to_json(), encoding="utf-8")
        print(f"✅ Google token cache updated: {token_path}")

    return creds


drive_creds = get_drive_credentials(
    client_secret_json_path=str(client_secret_path),
    token_json_path=str(token_cache_path),
    scopes=DRIVE_SCOPES,
)

drive_service = build("drive", "v3", credentials=drive_creds)
print("✅ Google Drive API client initialized")


# ------------------------------------------------------------
# Drive: who am I? (safe check)
# ------------------------------------------------------------
def drive_whoami(service) -> Dict:
    return service.about().get(fields="user(emailAddress,displayName),storageQuota").execute()

try:
    me = drive_whoami(drive_service)
    email = me.get("user", {}).get("emailAddress", "(unknown)")
    name = me.get("user", {}).get("displayName", "(unknown)")
    print(f"✅ Authorized Google account: {name} <{email}>")
except Exception as e:
    print("⚠️ Could not fetch Drive about.get:", type(e).__name__, str(e))


# ------------------------------------------------------------
# Drive folder checks
# ------------------------------------------------------------
def drive_get_folder_metadata(service, folder_id: str) -> Dict:
    return service.files().get(
        fileId=folder_id,
        fields="id,name,mimeType,trashed,driveId,owners(emailAddress),permissions",
        supportsAllDrives=True,
    ).execute()

def drive_list_folder_sample(service, folder_id: str, page_size: int = 5) -> List[Dict]:
    q = f"'{folder_id}' in parents and trashed=false"
    resp = service.files().list(
        q=q,
        pageSize=page_size,
        fields="files(id,name,mimeType,modifiedTime,owners(emailAddress))",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True,
    ).execute()
    return resp.get("files", [])


try:
    folder_meta = drive_get_folder_metadata(drive_service, DRIVE_FOLDER_ID)

    if folder_meta.get("trashed"):
        raise ValueError("Target Drive folder is in trash. Please restore it or change DRIVE_FOLDER_ID.")

    if folder_meta.get("mimeType") != "application/vnd.google-apps.folder":
        raise ValueError("DRIVE_FOLDER_ID does not point to a folder. Please set a folder ID.")

    print(f"✅ Drive folder access OK: {folder_meta.get('name')} (id={folder_meta.get('id')})")

    sample_files = drive_list_folder_sample(drive_service, DRIVE_FOLDER_ID, page_size=5)
    print(f"✅ Drive list OK: {len(sample_files)} sample file(s) found in target folder")

except HttpError as e:
    # 404 is often caused by permission/scope issues even if the ID is correct.
    if getattr(e, "resp", None) is not None and e.resp.status == 404:
        print("❌ Drive folder not accessible (404). Common causes:")
        print("  1) DRIVE_FOLDER_ID is wrong (not a folder ID, or copied incorrectly)")
        print("  2) The authorized Google account does not have permission to that folder")
        print("  3) Using 'drive.file' scope: existing folders/files may not be visible to the app")
        print("     -> Recommended fix: use scope 'https://www.googleapis.com/auth/drive' and re-auth")
        print("     -> Also delete google_token.json when changing scopes")
        print("  4) The folder is in a Shared Drive and access is missing")
        raise
    else:
        print("❌ Drive connectivity check failed:", e)
        raise

except Exception as e:
    print("❌ Drive connectivity check failed:", type(e).__name__, str(e))
    raise


# ------------------------------------------------------------
# Notion: connectivity checks (database metadata)
# ------------------------------------------------------------
def notion_get_database(database_id: str) -> Dict:
    url = f"https://api.notion.com/v1/databases/{database_id}"
    r = requests.get(url, headers=NOTION_HEADERS, timeout=30)
    if r.status_code != 200:
        raise ValueError(f"Notion DB access failed: {r.status_code} {r.text[:200]}")
    return r.json()

try:
    db_meta = notion_get_database(NOTION_DB_ID)
    title_parts = db_meta.get("title", [])
    db_title = "".join([t.get("plain_text", "") for t in title_parts]) if title_parts else "(untitled)"
    print(f"✅ Notion DB access OK: {db_title} (id={NOTION_DB_ID})")
except Exception as e:
    print("❌ Notion connectivity check failed:", type(e).__name__, str(e))
    raise


print("\n=== Readiness Summary ===")
print("Drive client:", "OK")
print("Notion auth:", "OK")
print("=========================\n")


Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=750875982200-85rnsoqhr2af2b13peueev0bm60q22sh.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A58025%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&state=nqc3PjgWJii4jKiyZgzYwVzPDWng2A&access_type=offline
✅ Google token cache updated: google_token.json
✅ Google Drive API client initialized
✅ Authorized Google account: Keisuke Nakatsuka <tokyo04hockey@gmail.com>
✅ Drive folder access OK: Reference (id=1SygzpVjCuk-_8oHk9XQOponn7T3ZOsgh)
✅ Drive list OK: 5 sample file(s) found in target folder
✅ Notion DB access OK: Research Question (id=2a98e0e4d16280b7bad2cf6635a3ef17)

=== Readiness Summary ===
Drive client: OK
Notion auth: OK



In [35]:
# ============================================================
# Cell 04: Load candidate papers and select processing targets (revised)
#  - Auto-pick latest 018 review CSV from filename timestamps
#  - REVIEW_PREFIX="review_top" (supports review_top{N}_YYYYMMDD_HHMMSS.csv)
#  - Normalize schema for downstream (019)
#  - Load idempotency history (Drive/Notion processed keys)
#  - Select processing targets (filters + MAX_PER_RUN)
# ============================================================

from __future__ import annotations

import os
import re
import json
import pathlib
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple

import pandas as pd

# ------------------------------------------------------------
# Configuration (override via env.txt if desired)
# ------------------------------------------------------------

# 018 candidates directory (where review_top*.csv files are saved)
BASE_018_CANDIDATES_DIR = pathlib.Path(
    os.getenv(
        "BASE_018_CANDIDATES_DIR",
        "./artifacts/018_seed_corpus_2023plus_discovery/candidates"
    )
)

# If set, use this exact file; otherwise auto-select the latest one.
ENV_CANDIDATES_PATH = os.getenv("CANDIDATES_PATH", "").strip()

# Pick latest file that matches: review_top{N}_YYYYMMDD_HHMMSS.csv
REVIEW_PREFIX = os.getenv("REVIEW_PREFIX", "review_top").strip()  # requested: "review_top"

# Filtering policy
YEAR_FROM = int(os.getenv("YEAR_FROM", "2023"))

PRIORITY_TIER_ALLOW = [x.strip() for x in os.getenv("PRIORITY_TIER_ALLOW", "P1,P2").split(",") if x.strip()]
RQ_RELEVANCE_ALLOW = [x.strip() for x in os.getenv("RQ_RELEVANCE_ALLOW", "HIGH,MEDIUM").split(",") if x.strip()]
FREE_PDF_ALLOW = [x.strip() for x in os.getenv("FREE_PDF_ALLOW", "HIGH").split(",") if x.strip()]

MAX_PER_RUN = int(os.getenv("MAX_PER_RUN", "50"))

# 019 idempotency / history location (optional)
HISTORY_DIR = pathlib.Path(os.getenv("HISTORY_DIR", "./artifacts/019_history"))
HISTORY_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_HISTORY_PATH = HISTORY_DIR / "drive_processed_keys.jsonl"
NOTION_HISTORY_PATH = HISTORY_DIR / "notion_processed_keys.jsonl"

# If true, ignore history and reprocess everything that passes filters
IGNORE_HISTORY = os.getenv("IGNORE_HISTORY", "false").lower() in ("1", "true", "yes")

# ------------------------------------------------------------
# Print config
# ------------------------------------------------------------
print("=== Candidate Loading Config (019) ===")
print("BASE_018_CANDIDATES_DIR:", BASE_018_CANDIDATES_DIR)
print("REVIEW_PREFIX:", REVIEW_PREFIX)
print("CANDIDATES_PATH (env override):", ENV_CANDIDATES_PATH if ENV_CANDIDATES_PATH else "(auto)")
print("YEAR_FROM:", YEAR_FROM)
print("PRIORITY_TIER_ALLOW:", PRIORITY_TIER_ALLOW)
print("RQ_RELEVANCE_ALLOW:", RQ_RELEVANCE_ALLOW)
print("FREE_PDF_ALLOW:", FREE_PDF_ALLOW)
print("MAX_PER_RUN:", MAX_PER_RUN)
print("IGNORE_HISTORY:", IGNORE_HISTORY)
print("HISTORY_DIR:", HISTORY_DIR)
print("=====================================\n")


# ------------------------------------------------------------
# Helpers: pick the latest 018 review file by timestamp in filename
# ------------------------------------------------------------
def _parse_ts_from_review_filename(name: str) -> Optional[datetime]:
    """
    Expected filename format (flexible topN):
      review_top{N}_YYYYMMDD_HHMMSS.csv
    e.g.
      review_top100_20260112_050231.csv

    We accept any prefix that starts with REVIEW_PREFIX (e.g. "review_top")
    as long as it ends with _YYYYMMDD_HHMMSS.csv.
    """
    # Capture the final timestamp chunk
    m = re.search(r"_(\d{8}_\d{6})\.csv$", name)
    if not m:
        return None
    try:
        return datetime.strptime(m.group(1), "%Y%m%d_%H%M%S")
    except ValueError:
        return None


def find_latest_review_csv(dir_path: pathlib.Path, prefix: str) -> pathlib.Path:
    """
    Find the latest file in dir_path that:
      - is a CSV
      - filename starts with prefix (e.g., 'review_top')
      - ends with _YYYYMMDD_HHMMSS.csv
    """
    if not dir_path.exists():
        raise FileNotFoundError(f"018 candidates dir not found: {dir_path}")

    candidates: List[Tuple[datetime, pathlib.Path]] = []
    for p in dir_path.iterdir():
        if not p.is_file():
            continue
        if p.suffix.lower() != ".csv":
            continue
        if not p.name.startswith(prefix):
            continue
        ts = _parse_ts_from_review_filename(p.name)
        if ts is None:
            continue
        candidates.append((ts, p))

    if not candidates:
        raise FileNotFoundError(
            f"No review CSV matched under: {dir_path}\n"
            f"Expected pattern: {prefix}*_*YYYYMMDD_HHMMSS.csv (e.g., review_top100_20260112_050231.csv)"
        )

    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]


# ------------------------------------------------------------
# Resolve CANDIDATES_PATH
# ------------------------------------------------------------
if ENV_CANDIDATES_PATH:
    CANDIDATES_PATH = pathlib.Path(ENV_CANDIDATES_PATH)
    print(f"ℹ️ Using CANDIDATES_PATH from env: {CANDIDATES_PATH}")
else:
    CANDIDATES_PATH = find_latest_review_csv(BASE_018_CANDIDATES_DIR, REVIEW_PREFIX)
    print(f"✅ Auto-selected latest 018 review CSV: {CANDIDATES_PATH}")

if not CANDIDATES_PATH.exists():
    raise FileNotFoundError(f"CANDIDATES_PATH does not exist: {CANDIDATES_PATH}")

# ------------------------------------------------------------
# Load candidates
# ------------------------------------------------------------
candidates_df = pd.read_csv(CANDIDATES_PATH)
print(f"\n✅ Loaded candidates: {len(candidates_df)} rows")


# ------------------------------------------------------------
# Normalize schema (make 019 downstream consistent)
# ------------------------------------------------------------
def _safe_str(x) -> str:
    if pd.isna(x) or x is None:
        return ""
    return str(x).strip()

def _safe_int(x, default=None):
    try:
        if pd.isna(x) or x is None:
            return default
        return int(float(x))
    except Exception:
        return default

# Map possible column aliases to canonical names
ALIASES = {
    "paper_id": ["paper_id", "id", "paper_key"],
    "year": ["publication_year", "year"],
    "title": ["title", "paper_title", "Name"],
    "venue": ["venue", "source", "journal"],
    "authors": ["authors", "author", "author_list"],
    "doi": ["doi", "DOI"],
    "landing_url": ["landing_page_url", "landing_url", "url", "URL"],
    "pdf_candidate_url": ["pdf_candidate_url", "pdf_url", "pdf", "PDF"],
    "priority_tier": ["priority_tier"],
    "priority_score": ["priority_score"],
    "rq_relevance_label": ["rq_relevance_label"],
    "free_pdf_label": ["free_pdf_label"],
}

def _pick_col(df: pd.DataFrame, names: List[str]) -> Optional[str]:
    for n in names:
        if n in df.columns:
            return n
    return None

colmap = {k: _pick_col(candidates_df, v) for k, v in ALIASES.items()}

def _get(row, key: str):
    c = colmap.get(key)
    return row[c] if c else None

norm_rows = []
for _, r in candidates_df.iterrows():
    year = _safe_int(_get(r, "year"))
    doi = _safe_str(_get(r, "doi"))
    landing_url = _safe_str(_get(r, "landing_url"))
    pdf_candidate_url = _safe_str(_get(r, "pdf_candidate_url"))

    # Canonical paper_id strategy:
    # - Prefer DOI
    # - Else fall back to landing_url (if exists)
    # - Else a synthetic key from title+year
    if doi:
        paper_id = f"doi:{doi}"
    elif landing_url:
        paper_id = f"url:{landing_url}"
    else:
        paper_id = f"synthetic:{_safe_str(_get(r,'title'))[:80]}:{year or 'nd'}"

    norm_rows.append({
        "paper_id": paper_id,
        "year": year,
        "title": _safe_str(_get(r, "title")),
        "venue": _safe_str(_get(r, "venue")),
        "authors": _safe_str(_get(r, "authors")),
        "doi": doi,
        "landing_url": landing_url,
        "pdf_candidate_url": pdf_candidate_url,
        "priority_tier": _safe_str(_get(r, "priority_tier")) or "P3",
        "priority_score": float(_get(r, "priority_score")) if _get(r, "priority_score") not in (None, "") else None,
        "rq_relevance_label": _safe_str(_get(r, "rq_relevance_label")),
        "free_pdf_label": _safe_str(_get(r, "free_pdf_label")),
    })

candidates_norm_df = pd.DataFrame(norm_rows)

print("\n✅ Normalized schema ready. Example keys:")
display(candidates_norm_df[["paper_id", "year", "title"]].head(3))


# ------------------------------------------------------------
# Load idempotency / history (Drive / Notion processed keys)
# ------------------------------------------------------------
def load_jsonl_keys(path: pathlib.Path) -> set:
    """
    Reads JSONL lines like {"paper_id": "...", "ts": "..."} and returns a set of paper_id.
    If file does not exist, returns empty set.
    """
    if not path.exists():
        return set()
    keys = set()
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                pid = str(obj.get("paper_id", "")).strip()
                if pid:
                    keys.add(pid)
            except Exception:
                continue
    return keys

drive_processed = set() if IGNORE_HISTORY else load_jsonl_keys(DRIVE_HISTORY_PATH)
notion_processed = set() if IGNORE_HISTORY else load_jsonl_keys(NOTION_HISTORY_PATH)
processed_union = drive_processed | notion_processed

print("\n=== Idempotency / History ===")
print("Drive processed keys: ", len(drive_processed))
print("Notion processed keys:", len(notion_processed))
print("Total processed (Drive ∪ Notion):", len(processed_union))
print("=============================\n")


# ------------------------------------------------------------
# Apply filters and select targets
# ------------------------------------------------------------
f = candidates_norm_df.copy()

# Filter: year
if YEAR_FROM:
    f = f[(f["year"].fillna(0).astype(int) >= YEAR_FROM)]

# Filter: priority tier
if PRIORITY_TIER_ALLOW:
    f = f[f["priority_tier"].astype(str).isin(PRIORITY_TIER_ALLOW)]

# Filter: RQ relevance
if RQ_RELEVANCE_ALLOW:
    f = f[f["rq_relevance_label"].astype(str).isin(RQ_RELEVANCE_ALLOW)]

# Filter: free PDF label
if FREE_PDF_ALLOW:
    f = f[f["free_pdf_label"].astype(str).isin(FREE_PDF_ALLOW)]

# Exclude already processed (idempotency)
if processed_union:
    f = f[~f["paper_id"].astype(str).isin(processed_union)]

# Sort: priority_score desc if present else keep stable
if "priority_score" in f.columns and f["priority_score"].notna().any():
    f = f.sort_values(["priority_score"], ascending=False)
else:
    f = f.sort_values(["priority_tier", "year"], ascending=[True, False])

targets_df = f.head(MAX_PER_RUN).reset_index(drop=True)

print("=== Target Selection Summary ===")
print(f"Loaded: {len(candidates_norm_df)} | Filtered: {len(f)} | Selected: {len(targets_df)}")
print("priority_tier:", dict(targets_df["priority_tier"].value_counts()))
print("rq_relevance_label:", dict(targets_df["rq_relevance_label"].value_counts()))
print("free_pdf_label:", dict(targets_df["free_pdf_label"].value_counts()))
print("===============================\n")

# Optional preview for human inspection
display_cols = [
    "paper_id", "year", "priority_tier", "priority_score",
    "rq_relevance_label", "free_pdf_label", "title", "doi", "landing_url", "pdf_candidate_url"
]
display(targets_df[display_cols].head(20))

print("\n✅ Cell 04 ready: targets_df prepared for Cell 05+ (URL candidate construction and ingestion).")


=== Candidate Loading Config (019) ===
BASE_018_CANDIDATES_DIR: artifacts/018_seed_corpus_2023plus_discovery/candidates
REVIEW_PREFIX: review_top
CANDIDATES_PATH (env override): (auto)
YEAR_FROM: 2023
PRIORITY_TIER_ALLOW: ['P1', 'P2']
RQ_RELEVANCE_ALLOW: ['HIGH', 'MEDIUM']
FREE_PDF_ALLOW: ['HIGH']
MAX_PER_RUN: 50
IGNORE_HISTORY: False
HISTORY_DIR: artifacts/019_history

✅ Auto-selected latest 018 review CSV: artifacts/018_seed_corpus_2023plus_discovery/candidates/review_top100_20260112_050231.csv

✅ Loaded candidates: 100 rows

✅ Normalized schema ready. Example keys:


,paper_id,year,title
0,doi:10.1002/sej.1515,2024,Venture capital exit after venture IPO
1,doi:10.1016/j.jbusvent.2023.106345,2023,"Scalability, venture capital availability, and..."
2,doi:10.1111/1467-8551.12803,2024,Post‐IPO lead venture capital firm involvement...



=== Idempotency / History ===
Drive processed keys:  0
Notion processed keys: 0
Total processed (Drive ∪ Notion): 0

=== Target Selection Summary ===
Loaded: 100 | Filtered: 87 | Selected: 50
priority_tier: {'P2': 45, 'P1': 5}
rq_relevance_label: {'MEDIUM': 30, 'HIGH': 20}
free_pdf_label: {'HIGH': 50}



,paper_id,year,priority_tier,priority_score,rq_relevance_label,free_pdf_label,title,doi,landing_url,pdf_candidate_url
0,doi:10.1002/sej.1515,2024,P1,70.40,HIGH,HIGH,Venture capital exit after venture IPO,10.1002/sej.1515,https://doi.org/10.1002/sej.1515,https://onlinelibrary.wiley.com/doi/pdfdirect/...
1,doi:10.1111/1467-8551.12803,2024,P1,68.12,HIGH,HIGH,Post‐IPO lead venture capital firm involvement...,10.1111/1467-8551.12803,https://doi.org/10.1111/1467-8551.12803,https://onlinelibrary.wiley.com/doi/pdfdirect/...
2,doi:10.1111/jfir.12412,2024,P1,67.12,HIGH,HIGH,"VC ownership post‐IPO: When, why, and how do V...",10.1111/jfir.12412,https://doi.org/10.1111/jfir.12412,https://onlinelibrary.wiley.com/doi/pdfdirect/...
3,url:https://arxiv.org/abs/2601.00810,2025,P1,66.69,HIGH,HIGH,Can Large Language Models Improve Venture Capi...,,https://arxiv.org/abs/2601.00810,https://arxiv.org/pdf/2601.00810
4,doi:10.30574/wjarr.2024.22.1.1047,2024,P1,65.16,MEDIUM,HIGH,The role of policy and regulation in promoting...,10.30574/wjarr.2024.22.1.1047,https://doi.org/10.30574/wjarr.2024.22.1.1047,https://wjarr.com/sites/default/files/WJARR-20...
5,doi:10.1007/s43441-025-00773-3,2025,P2,64.06,HIGH,HIGH,The Inflation Reduction Act’s Impact Upon Earl...,10.1007/s43441-025-00773-3,https://doi.org/10.1007/s43441-025-00773-3,https://link.springer.com/content/pdf/10.1007/...
6,doi:10.1146/annurev-financial-111021-100657,2023,P2,64.02,MEDIUM,HIGH,IPOs and SPACs: Recent Developments,10.1146/annurev-financial-111021-100657,https://doi.org/10.1146/annurev-financial-1110...,https://www.annualreviews.org/doi/pdf/10.1146/...
7,doi:10.48550/arxiv.2307.03718,2023,P2,63.52,MEDIUM,HIGH,Frontier AI Regulation: Managing Emerging Risk...,10.48550/arxiv.2307.03718,https://arxiv.org/abs/2307.03718,https://arxiv.org/pdf/2307.03718
8,doi:10.1093/rfs/hhad071,2023,P2,63.47,MEDIUM,HIGH,Common Venture Capital Investors and Startup G...,10.1093/rfs/hhad071,https://doi.org/10.1093/rfs/hhad071,https://academic.oup.com/rfs/advance-article-p...
9,doi:10.36948/ijfmr.2025.v07i03.46824,2025,P2,63.24,HIGH,HIGH,UNDERSTANDING THE ROLE OF VENTURE CAPITAL BACK...,10.36948/ijfmr.2025.v07i03.46824,https://doi.org/10.36948/ijfmr.2025.v07i03.46824,https://www.ijfmr.com/papers/2025/3/46824.pdf



✅ Cell 04 ready: targets_df prepared for Cell 05+ (URL candidate construction and ingestion).


In [12]:
# ============================================================
# Cell 05: Construct ordered PDF URL candidates for each paper
# ============================================================
#
# This cell builds an ordered list of PDF URL candidates for each target paper.
# The list is used downstream by the downloader (Cell 06/07).
#
# Strategy (high-level):
# 1) Use best_pdf_url from 018 when it looks like a direct PDF link (highest confidence)
# 2) If DOI is present, add DOI resolver URLs and potential publisher PDF patterns
# 3) Use landing_url as a fallback (may require HTML parsing in later steps)
# 4) Add lightweight heuristic-derived variants (e.g., arXiv pdf/abs swap)
#
# Outputs:
# - targets_df["pdf_url_candidates"] : List[str] (ordered, de-duplicated)
# - targets_df["pdf_url_primary"]    : First candidate (or None)
# - targets_df["pdf_url_notes"]      : Short notes about how the candidate set was built
#

import re
from urllib.parse import urlparse, urlunparse

# ------------------------------------------------------------
# URL helpers
# ------------------------------------------------------------
def normalize_url(url: str) -> str:
    """
    Normalize URL to reduce duplicates (e.g., strip whitespace, remove fragments).
    Does not aggressively remove query parameters because some publishers require them.
    """
    if url is None:
        return ""
    u = str(url).strip()
    if not u or u.lower() in ("none", "nan"):
        return ""
    try:
        p = urlparse(u)
        # Remove URL fragment (e.g., #page=1)
        p = p._replace(fragment="")
        return urlunparse(p)
    except Exception:
        return u


def looks_like_pdf_url(url: str) -> bool:
    """
    Heuristic: URL likely points to a PDF if it ends with .pdf,
    contains common pdf markers, or is an arXiv PDF link.
    """
    u = url.lower()
    return (
        u.endswith(".pdf")
        or "/pdf" in u
        or "pdfdirect" in u
        or "article-pdf" in u
        or "download=1" in u
        or "arxiv.org/pdf/" in u
    )


def doi_to_resolvers(doi: str) -> List[str]:
    """
    Generate DOI-based resolver URLs.
    These may redirect to a landing page or to a PDF depending on publisher.
    """
    d = (doi or "").strip()
    if not d or d.lower() in ("none", "nan"):
        return []
    d = d.replace("https://doi.org/", "").replace("http://doi.org/", "")
    return [
        f"https://doi.org/{d}",
        f"http://dx.doi.org/{d}",
    ]


def arxiv_variants(url: str) -> List[str]:
    """
    If URL is arXiv abs link, add pdf link; if pdf link, add abs link.
    """
    u = url.strip()
    if "arxiv.org/abs/" in u:
        return [u.replace("arxiv.org/abs/", "arxiv.org/pdf/")]
    if "arxiv.org/pdf/" in u:
        v = u.replace("arxiv.org/pdf/", "arxiv.org/abs/")
        v = re.sub(r"\.pdf$", "", v)
        return [v]
    return []


# ------------------------------------------------------------
# Publisher-specific lightweight patterns (optional but useful)
# ------------------------------------------------------------
def publisher_pdf_variants(best_pdf_url: str, landing_url: str, doi: str) -> List[str]:
    """
    Add a few conservative publisher URL variants.
    Keep this lightweight; deeper HTML-based discovery should happen later.
    """
    urls = []
    b = normalize_url(best_pdf_url)
    l = normalize_url(landing_url)

    # Wiley: often "pdfdirect" works only with cookies, but keep as candidate
    if "onlinelibrary.wiley.com/doi/" in l and "/pdfdirect" not in b:
        # Try to create pdfdirect from landing
        # Example: https://onlinelibrary.wiley.com/doi/10.1002/sej.1515 -> .../doi/pdfdirect/10.1002/sej.1515
        m = re.search(r"onlinelibrary\.wiley\.com/doi/(10\.\d{4,9}/[^?#]+)", l)
        if m:
            urls.append(f"https://onlinelibrary.wiley.com/doi/pdfdirect/{m.group(1)}")

    # OUP example: article-pdf links are often already in best_pdf_url; keep as-is
    # ScienceDirect: direct PDF often requires auth; DOI link might still be useful

    # Add DOI resolver (landing)
    urls.extend(doi_to_resolvers(doi))

    return [u for u in urls if u]


# ------------------------------------------------------------
# Build ordered candidates per row
# ------------------------------------------------------------
def build_pdf_candidates(row: pd.Series) -> Tuple[List[str], str]:
    """
    Build an ordered list of PDF URL candidates for a paper.
    Returns (candidate_list, notes).
    """
    candidates = []
    notes = []

    best_pdf = normalize_url(row.get("best_pdf_url", ""))
    landing = normalize_url(row.get("landing_url", ""))
    doi = str(row.get("doi") or "").strip()

    # 1) Primary: best_pdf_url (018's top guess)
    if best_pdf:
        candidates.append(best_pdf)
        notes.append("best_pdf_url")

        # Add arXiv variants if applicable (abs/pdf swap)
        candidates.extend(arxiv_variants(best_pdf))

    # 2) DOI resolvers + light publisher variants
    pub_vars = publisher_pdf_variants(best_pdf, landing, doi)
    if pub_vars:
        candidates.extend(pub_vars)
        notes.append("doi/publisher_variants")

    # 3) Landing URL as fallback (may require HTML parsing later)
    if landing:
        candidates.append(landing)
        notes.append("landing_url")

        # arXiv variants from landing too
        candidates.extend(arxiv_variants(landing))

    # 4) Conservative: if DOI exists, try a "content negotiation" style hint (still just URL)
    # (Actual content negotiation needs headers; downloader will handle it if implemented.)
    if doi and doi.lower() not in ("none", "nan"):
        candidates.append(f"https://doi.org/{doi.replace('https://doi.org/','').replace('http://doi.org/','')}")
        notes.append("doi_repeat_ok")

    # De-duplicate while preserving order
    seen = set()
    ordered = []
    for u in candidates:
        nu = normalize_url(u)
        if not nu:
            continue
        if nu not in seen:
            seen.add(nu)
            ordered.append(nu)

    # Optionally re-rank: put URLs that "look like PDF" first, but keep stability
    pdf_like = [u for u in ordered if looks_like_pdf_url(u)]
    non_pdf_like = [u for u in ordered if not looks_like_pdf_url(u)]
    final = pdf_like + non_pdf_like

    note_str = ",".join(sorted(set(notes))) if notes else "none"
    return final, note_str


# ------------------------------------------------------------
# Apply to targets_df
# ------------------------------------------------------------
pdf_candidates_notes = targets_df.apply(build_pdf_candidates, axis=1)
targets_df["pdf_url_candidates"] = [x[0] for x in pdf_candidates_notes]
targets_df["pdf_url_notes"] = [x[1] for x in pdf_candidates_notes]
targets_df["pdf_url_primary"] = targets_df["pdf_url_candidates"].apply(lambda xs: xs[0] if xs else None)

print("✅ Constructed pdf_url_candidates for targets.")
print("Example (first 3 papers):")
for i in range(min(3, len(targets_df))):
    print("\n---")
    print("paper_id:", targets_df.loc[i, "paper_id"])
    print("title:", targets_df.loc[i, "title"])
    print("primary:", targets_df.loc[i, "pdf_url_primary"])
    print("candidates:")
    for u in targets_df.loc[i, "pdf_url_candidates"][:6]:
        print(" -", u)
    if len(targets_df.loc[i, "pdf_url_candidates"]) > 6:
        print("   ...")

# Quick sanity: how many have at least 1 candidate?
n_with_candidates = (targets_df["pdf_url_candidates"].apply(len) > 0).sum()
print(f"\n✅ Targets with >=1 URL candidate: {n_with_candidates}/{len(targets_df)}")

# End of Cell 05


✅ Constructed pdf_url_candidates for targets.
Example (first 3 papers):

---
paper_id: doi:10.1002/sej.1515
title: Venture capital exit after venture IPO
primary: https://onlinelibrary.wiley.com/doi/pdfdirect/10.1002/sej.1515
candidates:
 - https://onlinelibrary.wiley.com/doi/pdfdirect/10.1002/sej.1515
 - https://doi.org/10.1002/sej.1515
 - http://dx.doi.org/10.1002/sej.1515

---
paper_id: doi:10.1111/1467-8551.12803
title: Post‐IPO lead venture capital firm involvement, merger‐related litigation and target firm valuation
primary: https://onlinelibrary.wiley.com/doi/pdfdirect/10.1111/1467-8551.12803
candidates:
 - https://onlinelibrary.wiley.com/doi/pdfdirect/10.1111/1467-8551.12803
 - https://doi.org/10.1111/1467-8551.12803
 - http://dx.doi.org/10.1111/1467-8551.12803

---
paper_id: doi:10.1111/jfir.12412
title: VC ownership post‐IPO: When, why, and how do VCs exit?
primary: https://onlinelibrary.wiley.com/doi/pdfdirect/10.1111/jfir.12412
candidates:
 - https://onlinelibrary.wiley.com

In [23]:
# ============================================================
# Cell 06: PDF download function (streaming, validation, hashing)
# ============================================================
#
# Goals:
# - Download PDFs safely via streaming (avoid memory spikes)
# - Validate that the response "looks like a PDF" (content-type + magic bytes)
# - Enforce minimum size thresholds (avoid HTML error pages)
# - Compute sha256 for idempotency + auditing
# - Produce rich per-attempt diagnostics (status_code/content_type/bytes/error)
#
# Notes:
# - This cell does NOT bypass paywalls. 403/401/302-to-HTML will be recorded.
# - Use Cell 07 to orchestrate retries / fallback ordering across candidates.
#

from __future__ import annotations

import os
import re
import time
import json
import hashlib
import pathlib
import mimetypes
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any, List, Tuple

import requests

# ------------------------------------------------------------
# Config knobs (override via env.txt)
# ------------------------------------------------------------
PDF_TIMEOUT_SEC = float(os.getenv("PDF_TIMEOUT_SEC", "60"))
PDF_MAX_REDIRECTS = int(os.getenv("PDF_MAX_REDIRECTS", "10"))
PDF_MIN_BYTES = int(os.getenv("PDF_MIN_BYTES", "50000"))  # 50KB default
PDF_MAX_BYTES = int(os.getenv("PDF_MAX_BYTES", str(200 * 1024 * 1024)))  # 200MB safeguard
PDF_SLEEP_BETWEEN_ATTEMPTS_SEC = float(os.getenv("PDF_SLEEP_BETWEEN_ATTEMPTS_SEC", "1.0"))

# If some sites block "no UA", a realistic UA helps reduce false failures.
PDF_HEADERS = {
    "Accept": "application/pdf,application/octet-stream;q=0.9,*/*;q=0.8",
    "User-Agent": os.getenv(
        "PDF_USER_AGENT",
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122 Safari/537.36",
    ),
    "Accept-Language": "en-US,en;q=0.9,ja;q=0.8",
    "Connection": "keep-alive",
}

# Optional: proxies (if you use corporate proxies)
# Example: export HTTP_PROXY / HTTPS_PROXY in env.txt
PDF_PROXIES = {}
if os.getenv("HTTP_PROXY"):
    PDF_PROXIES["http"] = os.getenv("HTTP_PROXY")
if os.getenv("HTTPS_PROXY"):
    PDF_PROXIES["https"] = os.getenv("HTTPS_PROXY")

# Output dir (set elsewhere; fallback to ./downloads)
DOWNLOAD_DIR = os.getenv("DOWNLOAD_DIR", "./artifacts/019_pdf_downloads")
pathlib.Path(DOWNLOAD_DIR).mkdir(parents=True, exist_ok=True)

print("=== PDF Download Config ===")
print("DOWNLOAD_DIR:", DOWNLOAD_DIR)
print("PDF_TIMEOUT_SEC:", PDF_TIMEOUT_SEC)
print("PDF_MIN_BYTES:", PDF_MIN_BYTES)
print("PDF_MAX_BYTES:", PDF_MAX_BYTES)
print("===========================\n")


# ------------------------------------------------------------
# Data model for attempts (for structured logs)
# ------------------------------------------------------------
@dataclass
class DownloadAttempt:
    url: str
    final_url: Optional[str] = None
    status_code: Optional[int] = None
    content_type: Optional[str] = None
    content_length: Optional[int] = None
    bytes: int = 0
    error: str = ""
    elapsed_sec: Optional[float] = None

    def to_dict(self) -> Dict[str, Any]:
        d = asdict(self)
        # Ensure JSON-serializable
        return d


# ------------------------------------------------------------
# Helpers: hashing, basic sanitization, validation
# ------------------------------------------------------------
def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def sanitize_filename(name: str, max_len: int = 160) -> str:
    name = (name or "").strip()
    name = re.sub(r"[\\/:*?\"<>|]+", "_", name)
    name = re.sub(r"\s+", " ", name).strip()
    if len(name) > max_len:
        name = name[:max_len].rstrip()
    return name or "paper"

def is_pdf_content_type(content_type: Optional[str]) -> bool:
    if not content_type:
        return False
    ct = content_type.lower()
    return ("application/pdf" in ct) or ("application/octet-stream" in ct)

def sniff_pdf_magic(local_path: str) -> bool:
    try:
        with open(local_path, "rb") as f:
            head = f.read(5)
        return head == b"%PDF-"
    except Exception:
        return False

def safe_int(x: Optional[str]) -> Optional[int]:
    try:
        if x is None:
            return None
        return int(x)
    except Exception:
        return None


# ------------------------------------------------------------
# Core: streaming downloader (single URL)
# ------------------------------------------------------------
def download_pdf_streaming(
    url: str,
    out_path: str,
    session: Optional[requests.Session] = None,
    timeout_sec: float = PDF_TIMEOUT_SEC,
    min_bytes: int = PDF_MIN_BYTES,
    max_bytes: int = PDF_MAX_BYTES,
) -> Tuple[bool, DownloadAttempt]:
    """
    Attempt to download a single URL.
    Writes to out_path if successful and valid.

    Returns: (ok, attempt)
    """
    sess = session or requests.Session()
    attempt = DownloadAttempt(url=url)

    t0 = time.time()
    try:
        resp = sess.get(
            url,
            stream=True,
            timeout=timeout_sec,
            allow_redirects=True,
            headers=PDF_HEADERS,
            proxies=PDF_PROXIES or None,
        )
        attempt.final_url = resp.url
        attempt.status_code = resp.status_code
        attempt.content_type = resp.headers.get("Content-Type")
        attempt.content_length = safe_int(resp.headers.get("Content-Length"))

        # Non-200 => record and stop
        if resp.status_code != 200:
            attempt.error = f"http_status_{resp.status_code}"
            attempt.elapsed_sec = round(time.time() - t0, 3)
            return False, attempt

        # Content-Type sanity check (not sufficient alone, but helps)
        if not is_pdf_content_type(attempt.content_type):
            # Might still be a PDF even if mislabelled, so we do a small stream and magic sniff.
            # We'll proceed but fail if magic bytes aren't PDF.
            pass

        # Stream to disk with hard size cap
        tmp_path = out_path + ".part"
        total = 0
        with open(tmp_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 256):
                if not chunk:
                    continue
                f.write(chunk)
                total += len(chunk)
                if total > max_bytes:
                    attempt.error = f"too_large>{max_bytes}"
                    break

        attempt.bytes = total
        attempt.elapsed_sec = round(time.time() - t0, 3)

        # If we broke due to size cap
        if attempt.error.startswith("too_large"):
            try:
                pathlib.Path(tmp_path).unlink(missing_ok=True)
            except Exception:
                pass
            return False, attempt

        # Basic size check (avoid HTML paywall pages)
        if total < min_bytes:
            attempt.error = f"too_small<{min_bytes}"
            try:
                pathlib.Path(tmp_path).unlink(missing_ok=True)
            except Exception:
                pass
            return False, attempt

        # Magic bytes check (strong signal)
        if not sniff_pdf_magic(tmp_path):
            attempt.error = "not_pdf_magic"
            try:
                pathlib.Path(tmp_path).unlink(missing_ok=True)
            except Exception:
                pass
            return False, attempt

        # Success: finalize
        pathlib.Path(tmp_path).rename(out_path)
        return True, attempt

    except requests.exceptions.RequestException as e:
        # This includes HTTPError, Timeout, ConnectionError, SSLError, etc.
        # Critically: we preserve response details if available.
        resp = getattr(e, "response", None)
        if resp is not None:
            attempt.final_url = getattr(resp, "url", None)
            attempt.status_code = getattr(resp, "status_code", None)
            attempt.content_type = (getattr(resp, "headers", {}) or {}).get("Content-Type")
            attempt.content_length = safe_int((getattr(resp, "headers", {}) or {}).get("Content-Length"))
        attempt.error = f"{type(e).__name__}: {str(e)[:160]}"
        attempt.elapsed_sec = round(time.time() - t0, 3)
        return False, attempt

    except Exception as e:
        attempt.error = f"{type(e).__name__}: {str(e)[:160]}"
        attempt.elapsed_sec = round(time.time() - t0, 3)
        return False, attempt


# ------------------------------------------------------------
# Public: multi-candidate fetch (used by Cell 11)
# ------------------------------------------------------------
def fetch_pdf_from_candidates(
    paper_id: str,
    url_candidates: List[str],
    out_dir: str = DOWNLOAD_DIR,
    filename_hint: Optional[str] = None,
    max_attempts: int = 8,
    sleep_sec: float = PDF_SLEEP_BETWEEN_ATTEMPTS_SEC,
) -> Dict[str, Any]:
    """
    Try multiple URL candidates to fetch a PDF. Returns a structured result dict.

    Result schema:
      ok: bool
      paper_id: str
      chosen_url: str | None
      local_path: str | None
      bytes: int
      sha256: str | None
      attempts: List[dict]
      final_error_code: str | None
      final_error_message: str | None
    """
    # De-dup while preserving order
    seen = set()
    deduped: List[str] = []
    for u in url_candidates or []:
        u = (u or "").strip()
        if not u or u in seen:
            continue
        seen.add(u)
        deduped.append(u)

    deduped = deduped[:max_attempts]

    out_dir_p = pathlib.Path(out_dir)
    out_dir_p.mkdir(parents=True, exist_ok=True)

    # Decide output filename
    if filename_hint:
        base = sanitize_filename(filename_hint)
    else:
        base = sanitize_filename(paper_id.replace("doi:", "").replace("url:", ""))
    out_path = str(out_dir_p / f"{base}.pdf")

    sess = requests.Session()

    attempts: List[Dict[str, Any]] = []
    last_err = None

    for u in deduped:
        ok, att = download_pdf_streaming(
            url=u,
            out_path=out_path,
            session=sess,
        )
        attempts.append(att.to_dict())

        if ok:
            sha = sha256_file(out_path)
            size = pathlib.Path(out_path).stat().st_size

            return {
                "ok": True,
                "paper_id": paper_id,
                "chosen_url": u,
                "local_path": out_path,
                "bytes": size,
                "sha256": sha,
                "attempts": attempts,
                "final_error_code": None,
                "final_error_message": None,
            }

        last_err = att.error
        time.sleep(sleep_sec)

    # Classify failure reason (helpful for ops)
    any_403 = any((a.get("status_code") == 403) for a in attempts)
    any_401 = any((a.get("status_code") == 401) for a in attempts)
    any_404 = any((a.get("status_code") == 404) for a in attempts)
    any_not_pdf = any((a.get("error") in ("not_pdf_magic", "not_pdf_content_type")) for a in attempts)

    if any_403 or any_401:
        final_code = "paywall_suspected"
    elif any_404:
        final_code = "not_found"
    elif any_not_pdf:
        final_code = "not_pdf"
    else:
        final_code = "fetch_failed"

    return {
        "ok": False,
        "paper_id": paper_id,
        "chosen_url": None,
        "local_path": None,
        "bytes": 0,
        "sha256": None,
        "attempts": attempts,
        "final_error_code": final_code,
        "final_error_message": last_err or "no_successful_download",
    }


print("✅ Cell 06 ready: download_pdf_streaming(...) + fetch_pdf_from_candidates(...)")
print("   - Rich diagnostics: status_code/content_type/bytes/error per attempt")
print("   - Validation: size threshold + %PDF magic bytes + max size cap")


=== PDF Download Config ===
DOWNLOAD_DIR: ./artifacts/019_pdf_downloads
PDF_TIMEOUT_SEC: 60.0
PDF_MIN_BYTES: 50000
PDF_MAX_BYTES: 209715200

✅ Cell 06 ready: download_pdf_streaming(...) + fetch_pdf_from_candidates(...)
   - Rich diagnostics: status_code/content_type/bytes/error per attempt
   - Validation: size threshold + %PDF magic bytes + max size cap


In [14]:
# ============================================================
# Cell 07: Retry and fallback strategy for failed downloads
# ============================================================
#
# This cell defines the "try multiple URL candidates" logic with:
# - per-URL download + validation (Cell 06)
# - retries handled at the HTTP layer (http_get_with_retry)
# - ordered fallback across multiple URL candidates
# - structured failure classification for analysis
#
# Outputs:
# - A function `fetch_pdf_from_candidates(...)` returning a structured dict:
#     - ok (bool)
#     - paper_id
#     - chosen_url (if ok)
#     - local_path, bytes, sha256 (if ok)
#     - attempts (list of per-URL attempt summaries)
#     - final_error_code / final_error_message (if not ok)
#
# Notes:
# - We keep this cell policy-focused: it orchestrates retries/fallback.
# - Deep alternative discovery (author page crawling, etc.) is handled later (Cell 08).
#

from typing import Any

# ------------------------------------------------------------
# Retry / fallback knobs (override via env.txt)
# ------------------------------------------------------------
MAX_URL_CANDIDATES = int(os.getenv("MAX_URL_CANDIDATES", "6"))  # per paper
MAX_TOTAL_ATTEMPTS = int(os.getenv("MAX_TOTAL_ATTEMPTS", "8"))  # hard safety cap
SLEEP_BETWEEN_URLS_SEC = float(os.getenv("SLEEP_BETWEEN_URLS_SEC", "0.5"))

# If True, allow re-trying the same URL candidate if it failed with transient errors
RETRY_TRANSIENT_URLS = os.getenv("RETRY_TRANSIENT_URLS", "true").lower() in ("1", "true", "yes")

# Error codes we consider transient (safe to try another URL and/or retry)
TRANSIENT_ERROR_CODES = set(
    x.strip().lower()
    for x in os.getenv("TRANSIENT_ERROR_CODES", "timeout,request_error,http_error").split(",")
    if x.strip()
)

# Treat these errors as "definitely not a PDF" for this URL (skip immediately)
NON_PDF_ERROR_CODES = set(
    x.strip().lower()
    for x in os.getenv("NON_PDF_ERROR_CODES", "not_pdf,too_small").split(",")
    if x.strip()
)

print("=== Download Fallback Policy ===")
print("MAX_URL_CANDIDATES:", MAX_URL_CANDIDATES)
print("MAX_TOTAL_ATTEMPTS:", MAX_TOTAL_ATTEMPTS)
print("SLEEP_BETWEEN_URLS_SEC:", SLEEP_BETWEEN_URLS_SEC)
print("RETRY_TRANSIENT_URLS:", RETRY_TRANSIENT_URLS)
print("TRANSIENT_ERROR_CODES:", sorted(list(TRANSIENT_ERROR_CODES)))
print("NON_PDF_ERROR_CODES:", sorted(list(NON_PDF_ERROR_CODES)))
print("===============================\n")


# ------------------------------------------------------------
# Orchestrator: try URL candidates in order
# ------------------------------------------------------------
def fetch_pdf_from_candidates(
    paper_id: str,
    url_candidates: List[str],
    out_dir: str = DOWNLOAD_DIR,
) -> Dict[str, Any]:
    """
    Try multiple URL candidates (in order) to fetch a valid PDF.

    Parameters
    ----------
    paper_id : str
        Stable identifier for the paper (doi:..., url:..., etc.).
    url_candidates : list[str]
        Ordered list of URLs to try.
    out_dir : str
        Directory to store temporary downloaded files.

    Returns
    -------
    dict
        Structured result with attempt logs.
    """
    attempts = []
    if not url_candidates:
        return {
            "ok": False,
            "paper_id": paper_id,
            "chosen_url": None,
            "local_path": None,
            "bytes": 0,
            "sha256": None,
            "attempts": attempts,
            "final_error_code": "no_url_candidates",
            "final_error_message": "No URL candidates were provided.",
        }

    # Limit candidates to avoid long runs
    candidates = [normalize_url(u) for u in url_candidates if normalize_url(u)]
    candidates = candidates[:MAX_URL_CANDIDATES]

    total_attempts = 0

    for url in candidates:
        if total_attempts >= MAX_TOTAL_ATTEMPTS:
            break

        # 1) Attempt download+validation for this URL
        res = download_pdf_with_validation(
            paper_id=paper_id,
            url=url,
            out_dir=out_dir,
            min_bytes=MIN_PDF_BYTES,
        )

        total_attempts += 1

        # Record attempt summary (keep it compact)
        attempts.append({
            "url": url,
            "ok": res["ok"],
            "http_status": res.get("http_status"),
            "bytes": res.get("bytes", 0),
            "error_code": (res.get("error_code") or ""),
            "error_message": (res.get("error_message") or "")[:200],
        })

        if res["ok"]:
            # Success
            return {
                "ok": True,
                "paper_id": paper_id,
                "chosen_url": url,
                "local_path": res["local_path"],
                "bytes": res["bytes"],
                "sha256": res["sha256"],
                "content_type": res.get("content_type"),
                "http_status": res.get("http_status"),
                "attempts": attempts,
                "final_error_code": None,
                "final_error_message": None,
            }

        # 2) Fallback decision logic
        err = (res.get("error_code") or "").lower()

        # If this URL returned HTML or too-small content, immediately move on.
        # If transient, optionally re-try same URL once (rarely helps, but can).
        if err in NON_PDF_ERROR_CODES:
            # Not a PDF for this URL; move to next candidate
            pass
        elif err in TRANSIENT_ERROR_CODES and RETRY_TRANSIENT_URLS:
            # Optional: a second attempt for transient errors, then move on
            if total_attempts < MAX_TOTAL_ATTEMPTS:
                time.sleep(SLEEP_BETWEEN_URLS_SEC)
                res2 = download_pdf_with_validation(
                    paper_id=paper_id,
                    url=url,
                    out_dir=out_dir,
                    min_bytes=MIN_PDF_BYTES,
                )
                total_attempts += 1
                attempts.append({
                    "url": url,
                    "ok": res2["ok"],
                    "http_status": res2.get("http_status"),
                    "bytes": res2.get("bytes", 0),
                    "error_code": (res2.get("error_code") or ""),
                    "error_message": (res2.get("error_message") or "")[:200],
                })
                if res2["ok"]:
                    return {
                        "ok": True,
                        "paper_id": paper_id,
                        "chosen_url": url,
                        "local_path": res2["local_path"],
                        "bytes": res2["bytes"],
                        "sha256": res2["sha256"],
                        "content_type": res2.get("content_type"),
                        "http_status": res2.get("http_status"),
                        "attempts": attempts,
                        "final_error_code": None,
                        "final_error_message": None,
                    }

        # Small delay between URL candidates to be polite and reduce rate limiting
        time.sleep(SLEEP_BETWEEN_URLS_SEC)

    # If we got here, everything failed
    last = attempts[-1] if attempts else {}
    return {
        "ok": False,
        "paper_id": paper_id,
        "chosen_url": None,
        "local_path": None,
        "bytes": 0,
        "sha256": None,
        "attempts": attempts,
        "final_error_code": (last.get("error_code") or "all_failed"),
        "final_error_message": (last.get("error_message") or "All URL candidates failed."),
    }


# ------------------------------------------------------------
# Optional: dry-run style evaluation (no network calls)
# ------------------------------------------------------------
# If you want a "policy preview" without downloading, implement it here.
# For now, we keep this cell focused on actual download fallback logic.

print("✅ fetch_pdf_from_candidates(...) is ready.")

# End of Cell 07


=== Download Fallback Policy ===
MAX_URL_CANDIDATES: 6
MAX_TOTAL_ATTEMPTS: 8
SLEEP_BETWEEN_URLS_SEC: 0.5
RETRY_TRANSIENT_URLS: True
TRANSIENT_ERROR_CODES: ['http_error', 'request_error', 'timeout']
NON_PDF_ERROR_CODES: ['not_pdf', 'too_small']

✅ fetch_pdf_from_candidates(...) is ready.


In [15]:
# ============================================================
# Cell 08: Optional alternative PDF discovery (private add-on hook)
# ============================================================
#
# This cell provides an OPTIONAL hook for a private add-on that attempts
# to discover alternative PDF URLs when all "obvious" candidates fail.
#
# Philosophy:
# - Keep any scraping / heavyweight logic out of the public notebook.
# - Define a clean interface (input -> list of candidate URLs with evidence).
# - If the private module is not available, safely degrade to "no alternatives".
#
# Typical alternative sources (implemented privately):
# - arXiv / ePrint mirrors
# - author homepages / institutional repositories
# - preprint servers / accepted manuscript PDFs
# - crossref / unpaywall-like resolution (if not already done in 018)
#
# Outputs:
# - `discover_alternative_pdf_urls(row, attempts)` function:
#     returns a list of URL candidates (ordered)
# - `extend_candidates_with_alternatives(row, attempts)` helper:
#     returns merged candidate list (original + new, de-duplicated)
#

from typing import Any

# Toggle: enable/disable alternative discovery
ENABLE_ALT_DISCOVERY = os.getenv("ENABLE_ALT_DISCOVERY", "false").lower() in ("1", "true", "yes")
print("ENABLE_ALT_DISCOVERY:", ENABLE_ALT_DISCOVERY)

# ------------------------------------------------------------
# Private module import (optional)
# ------------------------------------------------------------
# Your private add-on can live anywhere on PYTHONPATH, e.g.:
# - ./private_addons/pdf_discovery.py
# - a private pip package
#
# Expected function signature (example):
#   def find_alternatives(
#       title: str,
#       doi: Optional[str],
#       authors: Optional[str],
#       year: Optional[int],
#       venue: Optional[str],
#       landing_url: Optional[str],
#       context: Optional[dict] = None
#   ) -> List[dict]:
#       """
#       Returns list of candidates:
#         [{"url": "...", "source": "arxiv", "confidence": 0.8, "evidence": "..."}]
#       """
#
PRIVATE_DISCOVERY_AVAILABLE = False
_private_find_alternatives = None

try:
    # Example import path (customize to your repo layout)
    from private_addons.pdf_discovery import find_alternatives as _private_find_alternatives
    PRIVATE_DISCOVERY_AVAILABLE = True
    print("✅ Private add-on loaded: private_addons.pdf_discovery.find_alternatives")
except Exception as e:
    print("ℹ️ Private add-on not available (expected in public context).")
    print("   Reason:", type(e).__name__, str(e)[:200])


# ------------------------------------------------------------
# Public interface: discover alternatives (safe fallback if missing)
# ------------------------------------------------------------
def discover_alternative_pdf_urls(row: pd.Series, attempts: Optional[List[Dict[str, Any]]] = None) -> List[str]:
    """
    Discover alternative PDF URLs for a given paper row.

    Parameters
    ----------
    row : pd.Series
        A row from targets_df containing at least:
          - title, doi, authors, year, venue, landing_url
    attempts : list[dict], optional
        Attempt logs from fetch_pdf_from_candidates (for context/debugging).

    Returns
    -------
    list[str]
        Ordered list of alternative PDF URL candidates.
        Returns an empty list if discovery is disabled or private add-on is unavailable.
    """
    if not ENABLE_ALT_DISCOVERY:
        return []
    if not PRIVATE_DISCOVERY_AVAILABLE or _private_find_alternatives is None:
        return []

    title = str(row.get("title") or "").strip()
    doi = str(row.get("doi") or "").strip() or None
    authors = str(row.get("authors") or "").strip() or None
    year = row.get("year")
    year = int(year) if year is not None and str(year) != "nan" else None
    venue = str(row.get("venue") or "").strip() or None
    landing_url = str(row.get("landing_url") or "").strip() or None

    context = {
        "paper_id": str(row.get("paper_id") or ""),
        "attempts": attempts or [],
    }

    # Private add-on returns structured candidates; we extract URLs
    candidates_struct = _private_find_alternatives(
        title=title,
        doi=doi,
        authors=authors,
        year=year,
        venue=venue,
        landing_url=landing_url,
        context=context,
    )

    urls = []
    for c in (candidates_struct or []):
        u = normalize_url(c.get("url", ""))
        if u:
            urls.append(u)

    # De-duplicate preserving order
    seen = set()
    out = []
    for u in urls:
        if u not in seen:
            seen.add(u)
            out.append(u)

    return out


def extend_candidates_with_alternatives(
    row: pd.Series,
    attempts: Optional[List[Dict[str, Any]]] = None,
    max_total: int = 12,
) -> List[str]:
    """
    Merge existing pdf_url_candidates with newly discovered alternatives.

    Parameters
    ----------
    row : pd.Series
        Target row
    attempts : list[dict], optional
        Attempt logs (for context)
    max_total : int
        Hard cap of total candidates to prevent runaway processing

    Returns
    -------
    list[str]
        Merged ordered URL list
    """
    base = list(row.get("pdf_url_candidates") or [])
    alt = discover_alternative_pdf_urls(row, attempts=attempts)

    merged = []
    seen = set()
    for u in base + alt:
        nu = normalize_url(u)
        if not nu:
            continue
        if nu not in seen:
            seen.add(nu)
            merged.append(nu)

    return merged[:max_total]


print("✅ Alternative discovery hook is ready (may be disabled / unavailable).")

# End of Cell 08


ENABLE_ALT_DISCOVERY: False
ℹ️ Private add-on not available (expected in public context).
   Reason: ModuleNotFoundError No module named 'private_addons'
✅ Alternative discovery hook is ready (may be disabled / unavailable).


In [29]:
# ============================================================
# Cell 09: Google Drive upload and duplicate handling
# ============================================================
#
# This cell uploads downloaded PDFs to a target Google Drive folder and avoids duplicates.
#
# Duplicate handling strategy (recommended for your workflow):
# 1) List existing PDFs in the target Drive folder (once per run)
# 2) Pre-filter likely duplicates locally (cheap string similarity over filenames)
# 3) (Optional) Use ChatGPT API to judge whether a target paper already exists in Drive
#    - This helps with formatting differences / citation-style filenames
# 4) If duplicate is confirmed, skip upload and return the existing Drive file link/id
# 5) Otherwise, upload the PDF with a citation-like filename (Author, F. (Year). Title. Venue.pdf)
#
# Requirements:
# - drive_service + DRIVE_FOLDER_ID already initialized (Cell 03)
# - OPENAI_API_KEY available if LLM duplicate judgement is enabled (Cell 02)
# - targets_df contains metadata columns: title, authors, year, venue, doi (from Cell 04)
#
# Outputs:
# - drive_files_df: DataFrame of existing PDFs in the folder
# - helper functions:
#   - drive_list_pdfs_in_folder(...)
#   - format_drive_filename(...)
#   - prefilter_candidates_for_row(...)
#   - judge_duplicate_with_llm(...)
#   - drive_upload_pdf_with_dedupe(...)
#

import io
import re
import difflib
from typing import Any, Optional, List, Dict, Tuple
from urllib.parse import urlparse, urlunparse

from googleapiclient.http import MediaFileUpload
from googleapiclient.errors import HttpError


# ------------------------------------------------------------
# Config knobs (override via env.txt)
# ------------------------------------------------------------
DRIVE_LIST_PAGE_SIZE = int(os.getenv("DRIVE_LIST_PAGE_SIZE", "200"))
DRIVE_LIST_MAX = int(os.getenv("DRIVE_LIST_MAX", "5000"))  # safety cap

# Local prefiltering (cheap)
K_PREFILTER = int(os.getenv("K_PREFILTER", "15"))
PREFILTER_CUTOFF = float(os.getenv("PREFILTER_CUTOFF", "0.60"))  # difflib cutoff

# LLM duplicate judgement
LLM_DUPLICATE_ENABLED = os.getenv("LLM_DUPLICATE_ENABLED", "true").lower() in ("1", "true", "yes")
LLM_DUPLICATE_MODEL = os.getenv("LLM_DUPLICATE_MODEL", "gpt-4.1-mini")  # adjust to your available model
LLM_MAX_CANDIDATES_SENT = int(os.getenv("LLM_MAX_CANDIDATES_SENT", "10"))
LLM_TEMPERATURE = float(os.getenv("LLM_TEMPERATURE", "0.0"))
LLM_TIMEOUT_SEC = float(os.getenv("LLM_TIMEOUT_SEC", "30"))

# Upload behavior
DRIVE_UPLOAD_CHUNK_MB = int(os.getenv("DRIVE_UPLOAD_CHUNK_MB", "10"))
DRIVE_ALWAYS_UPLOAD_IF_NO_LLM = os.getenv("DRIVE_ALWAYS_UPLOAD_IF_NO_LLM", "true").lower() in ("1", "true", "yes")

print("=== Drive Dedupe Config ===")
print("DRIVE_LIST_PAGE_SIZE:", DRIVE_LIST_PAGE_SIZE)
print("DRIVE_LIST_MAX:", DRIVE_LIST_MAX)
print("K_PREFILTER:", K_PREFILTER)
print("PREFILTER_CUTOFF:", PREFILTER_CUTOFF)
print("LLM_DUPLICATE_ENABLED:", LLM_DUPLICATE_ENABLED)
print("LLM_DUPLICATE_MODEL:", LLM_DUPLICATE_MODEL)
print("LLM_MAX_CANDIDATES_SENT:", LLM_MAX_CANDIDATES_SENT)
print("DRIVE_UPLOAD_CHUNK_MB:", DRIVE_UPLOAD_CHUNK_MB)
print("===========================\n")


# ------------------------------------------------------------
# 1) List existing PDFs in the target folder (paged)
# ------------------------------------------------------------
def drive_list_pdfs_in_folder(service, folder_id: str, max_items: int = 5000) -> pd.DataFrame:
    """
    List PDF files in a Drive folder.
    Returns a DataFrame with metadata used for deduplication.
    """
    files: List[Dict[str, Any]] = []
    page_token = None
    fetched = 0

    q = f"'{folder_id}' in parents and trashed=false and mimeType='application/pdf'"

    while True:
        resp = service.files().list(
            q=q,
            pageSize=DRIVE_LIST_PAGE_SIZE,
            pageToken=page_token,
            fields="nextPageToken,files(id,name,mimeType,md5Checksum,size,modifiedTime,createdTime,webViewLink)",
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()

        batch = resp.get("files", [])
        files.extend(batch)
        fetched += len(batch)

        page_token = resp.get("nextPageToken")
        if not page_token or fetched >= max_items:
            break

    df = pd.DataFrame(files)
    if len(df) == 0:
        print("ℹ️ No existing PDFs found in the target Drive folder.")
        return df

    if "size" in df.columns:
        df["size"] = pd.to_numeric(df["size"], errors="coerce").fillna(0).astype(int)

    return df


drive_files_df = drive_list_pdfs_in_folder(drive_service, DRIVE_FOLDER_ID, max_items=DRIVE_LIST_MAX)
print(f"✅ Drive PDFs listed: {len(drive_files_df):,}")
display(drive_files_df.head(10))


# ------------------------------------------------------------
# 2) Filename formatting (your convention)
# ------------------------------------------------------------
def format_author_citation(authors: str) -> str:
    """
    Convert an authors string into a short citation-like author part.
    Heuristic: take first author and format as "Last, F."
    """
    if not authors:
        return "Unknown"
    a = str(authors).strip()

    # Common format from 018: "Yong Li, Tailan Chi, Sai Lan et al."
    first = a.split(",")[0].strip()

    parts = first.split()
    if len(parts) >= 2:
        last = parts[-1].strip()
        first_initial = parts[0].strip()[0].upper()
        return f"{last}, {first_initial}."
    return first


def _clean_filename_text(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s or "")).strip()
    # Replace problematic filename characters
    s = s.replace("/", "-").replace("\\", "-").replace(":", "-").replace("|", "-")
    return s


def format_drive_filename(row: pd.Series, max_len: int = 180) -> str:
    """
    Deterministic citation-like filename:
      "Author, F. (YEAR). Title. Venue.pdf"
    """
    author_part = format_author_citation(row.get("authors", ""))
    year = row.get("year")
    year_str = str(int(year)) if pd.notna(year) else "n.d."
    title = _clean_filename_text(row.get("title", ""))
    venue = _clean_filename_text(row.get("venue", ""))

    base = f"{author_part} ({year_str}). {title}."
    if venue:
        base += f" {venue}"

    base = base.strip()
    base = base[:max_len].rstrip(" .")
    return f"{base}.pdf"


# ------------------------------------------------------------
# 3) Local pre-filtering: pick likely duplicates by filename similarity
# ------------------------------------------------------------
def normalize_for_match(s: str) -> str:
    s = str(s or "").lower().strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s\(\)\.\-]", "", s)  # keep mild punctuation
    return s


def build_candidate_signature(row: pd.Series) -> str:
    author = str(row.get("authors") or "")
    year = row.get("year")
    year_str = str(int(year)) if pd.notna(year) else ""
    title = str(row.get("title") or "")
    venue = str(row.get("venue") or "")
    sig = f"{author} {year_str} {title} {venue}"
    return normalize_for_match(sig)


def build_drive_signature(filename: str) -> str:
    return normalize_for_match(filename)


if len(drive_files_df):
    drive_files_df["sig"] = drive_files_df["name"].apply(build_drive_signature)
else:
    drive_files_df["sig"] = pd.Series(dtype=str)

targets_df["sig"] = targets_df.apply(build_candidate_signature, axis=1)


def prefilter_candidates_for_row(
    row: pd.Series,
    drive_df: pd.DataFrame,
    k: int = 15,
    cutoff: float = 0.6,
) -> pd.DataFrame:
    """
    Return a small DataFrame of Drive file candidates likely to match this row.
    Uses difflib over precomputed signatures.
    """
    if len(drive_df) == 0:
        return drive_df.iloc[0:0]

    sig = row.get("sig") or ""
    choices = drive_df["sig"].tolist()

    # difflib returns matched strings; we map them back to rows by signature
    matched_sigs = difflib.get_close_matches(sig, choices, n=k, cutoff=cutoff)
    if not matched_sigs:
        return drive_df.iloc[0:0]

    # Keep original order by taking first occurrences
    mask = drive_df["sig"].isin(matched_sigs)
    out = drive_df[mask].copy()

    # Optional: prioritize exact DOI/year hints by filename presence (light heuristic)
    y = row.get("year")
    if pd.notna(y):
        y_str = str(int(y))
        out["year_hint"] = out["name"].astype(str).str.contains(y_str, regex=False).astype(int)
        out = out.sort_values(["year_hint", "modifiedTime"], ascending=[False, False])

    return out.head(k)


# ------------------------------------------------------------
# 4) LLM duplicate judgement (ChatGPT API) — robust interface
# ------------------------------------------------------------
def build_duplicate_prompt(paper_row: pd.Series, drive_candidates: pd.DataFrame) -> str:
    """
    Build a compact prompt for duplicate judgement.
    """
    paper = {
        "title": str(paper_row.get("title") or ""),
        "authors": str(paper_row.get("authors") or ""),
        "year": int(paper_row["year"]) if pd.notna(paper_row.get("year")) else None,
        "venue": str(paper_row.get("venue") or ""),
        "doi": str(paper_row.get("doi") or ""),
    }

    cand_list: List[Dict[str, Any]] = []
    for _, r in drive_candidates.iterrows():
        cand_list.append({
            "id": r.get("id"),
            "name": r.get("name"),
            "size": r.get("size"),
            "modifiedTime": r.get("modifiedTime"),
        })

    return f"""
You are a strict deduplication assistant.

Task:
Given a target paper metadata and a list of existing Google Drive PDF files (names + minimal metadata),
decide whether the same scholarly work already exists in the folder.

Matching rules:
- Consider it a duplicate only if it is the SAME scholarly work (same title and year; authors consistent).
- Ignore formatting differences (punctuation, abbreviations, citation style, minor typos).
- If uncertain, return is_duplicate=false.

Return ONLY valid JSON with keys:
- is_duplicate: boolean
- best_match_file_id: string or null
- confidence: number between 0 and 1
- reason: short string

Target paper:
{json.dumps(paper, ensure_ascii=False)}

Existing Drive candidates:
{json.dumps(cand_list, ensure_ascii=False)}
""".strip()


def call_llm_json(prompt: str) -> Dict[str, Any]:
    """
    Calls the OpenAI API and returns parsed JSON.
    This implementation supports the modern 'openai' SDK (OpenAI()).
    If your environment differs, replace this function only.
    """
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY is not set, but LLM_DUPLICATE_ENABLED=true")

    # Lazy import so the notebook can run even if openai isn't installed
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)

        # Responses API
        resp = client.responses.create(
            model=LLM_DUPLICATE_MODEL,
            input=prompt,
            temperature=LLM_TEMPERATURE,
            timeout=LLM_TIMEOUT_SEC,
        )

        # Extract text output
        text = ""
        if hasattr(resp, "output_text"):
            text = resp.output_text
        else:
            # Fallback: try to find text in output structure
            text = str(resp)

    except Exception as e:
        raise RuntimeError(f"OpenAI call failed: {type(e).__name__}: {str(e)[:200]}")

    text = (text or "").strip()

    # Parse JSON strictly (common failure is extra text; we attempt a safe extraction)
    try:
        return json.loads(text)
    except Exception:
        # Try to extract the first JSON object substring
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not m:
            raise ValueError(f"LLM did not return JSON. Got: {text[:200]}")
        return json.loads(m.group(0))


def judge_duplicate_with_llm(paper_row: pd.Series, drive_df: pd.DataFrame) -> Dict[str, Any]:
    """
    Return a JSON-like dict:
      {is_duplicate, best_match_file_id, confidence, reason}
    """
    cand_df = prefilter_candidates_for_row(
        paper_row,
        drive_df,
        k=K_PREFILTER,
        cutoff=PREFILTER_CUTOFF,
    )

    if len(cand_df) == 0:
        return {"is_duplicate": False, "best_match_file_id": None, "confidence": 0.0, "reason": "no_prefilter_hits"}

    cand_df = cand_df.head(LLM_MAX_CANDIDATES_SENT)
    prompt = build_duplicate_prompt(paper_row, cand_df)
    out = call_llm_json(prompt)

    # Normalize output defensively
    return {
        "is_duplicate": bool(out.get("is_duplicate", False)),
        "best_match_file_id": out.get("best_match_file_id", None),
        "confidence": float(out.get("confidence", 0.0) or 0.0),
        "reason": str(out.get("reason", ""))[:200],
    }


# ------------------------------------------------------------
# 5) Drive helpers: fetch metadata and upload
# ------------------------------------------------------------
def drive_get_file_metadata(service, file_id: str) -> Dict[str, Any]:
    return service.files().get(
        fileId=file_id,
        fields="id,name,webViewLink,md5Checksum,size,modifiedTime",
        supportsAllDrives=True,
    ).execute()


def drive_upload_pdf_file(
    service,
    folder_id: str,
    local_path: str,
    filename: str,
) -> Dict[str, Any]:
    """
    Upload a local PDF file to Drive.
    """
    file_metadata = {"name": filename, "parents": [folder_id]}
    media = MediaFileUpload(
        local_path,
        mimetype="application/pdf",
        resumable=True,
        chunksize=DRIVE_UPLOAD_CHUNK_MB * 1024 * 1024,
    )

    created = service.files().create(
        body=file_metadata,
        media_body=media,
        fields="id,name,webViewLink,md5Checksum",
        supportsAllDrives=True,
    ).execute()

    return created


# ------------------------------------------------------------
# 6) Orchestrator: upload with LLM-aware dedupe
# ------------------------------------------------------------
def drive_upload_pdf_with_dedupe(
    service,
    folder_id: str,
    local_path: str,
    paper_row: pd.Series,
    paper_id: str,
    local_sha256: Optional[str] = None,
    drive_df: Optional[pd.DataFrame] = None,
    llm_enabled: bool = True,
    llm_confidence_threshold: float = 0.85,
) -> Dict[str, Any]:
    """
    Upload a PDF unless it already exists in Drive (LLM-based judgement).

    Returns dict with:
      - status: uploaded | skipped_duplicate | failed
      - drive_file_id
      - drive_link
      - filename
      - reason
      - dedupe: {method, confidence, matched_file_id}
    """
    if not local_path or not pathlib.Path(local_path).exists():
        return {
            "status": "failed",
            "paper_id": paper_id,
            "drive_file_id": None,
            "drive_link": None,
            "filename": None,
            "reason": "Local file not found.",
            "dedupe": {"method": "none"},
        }

    filename = format_drive_filename(paper_row)

    # If no drive_df provided, skip LLM dedupe (or you can list again, but that's expensive)
    drive_df = drive_df if drive_df is not None else pd.DataFrame()

    # LLM-based duplicate judgement (recommended)
    if llm_enabled and LLM_DUPLICATE_ENABLED and len(drive_df) > 0:
        try:
            j = judge_duplicate_with_llm(paper_row, drive_df)
            if j["is_duplicate"] and j["best_match_file_id"] and j["confidence"] >= llm_confidence_threshold:
                meta = drive_get_file_metadata(service, j["best_match_file_id"])
                return {
                    "status": "skipped_duplicate",
                    "paper_id": paper_id,
                    "drive_file_id": meta.get("id"),
                    "drive_link": meta.get("webViewLink"),
                    "filename": meta.get("name"),
                    "reason": f"Duplicate detected by LLM (confidence={j['confidence']:.2f}): {j['reason']}",
                    "dedupe": {"method": "llm", "confidence": j["confidence"], "matched_file_id": meta.get("id")},
                    "local_sha256": local_sha256,
                }
        except Exception as e:
            # If LLM fails, fall back to upload (or you can choose to fail fast)
            if not DRIVE_ALWAYS_UPLOAD_IF_NO_LLM:
                return {
                    "status": "failed",
                    "paper_id": paper_id,
                    "drive_file_id": None,
                    "drive_link": None,
                    "filename": filename,
                    "reason": f"LLM dedupe failed and DRIVE_ALWAYS_UPLOAD_IF_NO_LLM=false: {type(e).__name__}: {str(e)[:200]}",
                    "dedupe": {"method": "llm_error"},
                }

    # Upload
    try:
        created = drive_upload_pdf_file(service, folder_id, local_path, filename)

        # Update in-memory drive_df so subsequent rows can dedupe against newly uploaded files
        # (This keeps the run consistent even if you upload multiple PDFs.)
        if drive_df is not None:
            new_row = {
                "id": created.get("id"),
                "name": created.get("name"),
                "mimeType": "application/pdf",
                "md5Checksum": created.get("md5Checksum"),
                "size": pathlib.Path(local_path).stat().st_size,
                "modifiedTime": datetime.utcnow().isoformat() + "Z",
                "createdTime": datetime.utcnow().isoformat() + "Z",
                "webViewLink": created.get("webViewLink"),
            }
            # Append safely
            global drive_files_df
            try:
                drive_files_df = pd.concat([drive_files_df, pd.DataFrame([new_row])], ignore_index=True)
                drive_files_df["sig"] = drive_files_df["name"].apply(build_drive_signature)
            except Exception:
                pass

        return {
            "status": "uploaded",
            "paper_id": paper_id,
            "drive_file_id": created.get("id"),
            "drive_link": created.get("webViewLink"),
            "filename": created.get("name"),
            "reason": None,
            "dedupe": {"method": "none"},
            "drive_md5": created.get("md5Checksum"),
            "local_sha256": local_sha256,
        }

    except Exception as e:
        return {
            "status": "failed",
            "paper_id": paper_id,
            "drive_file_id": None,
            "drive_link": None,
            "filename": filename,
            "reason": f"{type(e).__name__}: {str(e)[:200]}",
            "dedupe": {"method": "none"},
        }


# ------------------------------------------------------------
# Optional: pre-flight visibility (top 10 targets)
# ------------------------------------------------------------
PREVIEW_DUPLICATES = os.getenv("PREVIEW_DUPLICATES", "true").lower() in ("1", "true", "yes")

if PREVIEW_DUPLICATES and len(targets_df) > 0 and len(drive_files_df) > 0:
    print("\n🔎 Pre-flight duplicate preview (first 10 targets):")
    scan_n = min(10, len(targets_df))
    for i in range(scan_n):
        row = targets_df.iloc[i]
        pid = str(row["paper_id"])
        fname = format_drive_filename(row)
        cand = prefilter_candidates_for_row(row, drive_files_df, k=5, cutoff=PREFILTER_CUTOFF)
        cand_names = cand["name"].tolist()[:3] if len(cand) else []
        print(f"- {pid} | filename='{fname[:80]}...' | prefilter_hits={len(cand)} | sample={cand_names}")

print("\n✅ Cell 09 ready: Drive listing + LLM-aware dedupe + upload function.")
# End of Cell 09


=== Drive Dedupe Config ===
DRIVE_LIST_PAGE_SIZE: 200
DRIVE_LIST_MAX: 5000
K_PREFILTER: 15
PREFILTER_CUTOFF: 0.6
LLM_DUPLICATE_ENABLED: True
LLM_DUPLICATE_MODEL: gpt-4.1-mini
LLM_MAX_CANDIDATES_SENT: 10
DRIVE_UPLOAD_CHUNK_MB: 10

✅ Drive PDFs listed: 35


,id,name,mimeType,webViewLink,createdTime,modifiedTime,md5Checksum,size
0,1A83Ai1AX_nnVPsKYu_ESI26pTpIAygOP,"Lerner, J. (1999). The government as venture c...",application/pdf,https://drive.google.com/file/d/1A83Ai1AX_nnVP...,2026-01-02T13:57:57.463Z,2026-01-02T13:58:03.907Z,62af39bf1741ad3341aa7b752fb82b4b,125060
1,1O7_GY0k9T2Egcs6WBtCuGO2nsZttKUFU,Bremner__paper__20260102.pdf,application/pdf,https://drive.google.com/file/d/1O7_GY0k9T2Egc...,2026-01-02T13:52:38.649Z,2026-01-02T13:52:46.401Z,ff7f21fde7a94bb949a22a25ddeeff39,442526
2,1fZvs7CSRDnrG72PNys9Uk_XsCKkravXy,1-s2.0-S3050700625000817-main__paper__20260102...,application/pdf,https://drive.google.com/file/d/1fZvs7CSRDnrG7...,2026-01-02T13:34:35.124Z,2026-01-02T13:35:03.183Z,03e0e04c349274bd6d03ea0119f6a336,894468
3,1AReahkgowqrj6ig-Jzj3B42yjwJ-ceB2,Refugees-and-entrepreneurship-IBS-WP-07-2025__...,application/pdf,https://drive.google.com/file/d/1AReahkgowqrj6...,2026-01-02T13:27:41.972Z,2026-01-02T13:27:48.815Z,ba5ed80e0e5592daaec293738a158bac,39023468
4,1TPemKb9cNO1PW_u4br0sSjasjPjvIg6Q,2511.23364v1__paper__20260102.pdf,application/pdf,https://drive.google.com/file/d/1TPemKb9cNO1PW...,2026-01-02T13:01:34.208Z,2026-01-02T13:01:42.654Z,331262396db4d6b4e657d7daa971571a,319784
5,16sTGkoD8ARG4GnGBKu5vnpSAJrb_Zsll,"Lerner, J. (1999). The government as venture c...",application/pdf,https://drive.google.com/file/d/16sTGkoD8ARG4G...,2026-01-01T23:06:18.122Z,2026-01-01T23:53:10.569Z,62af39bf1741ad3341aa7b752fb82b4b,125060
6,1gwqTSpfg6LqfbtxRsi-R6-2PueHczEIG,Kariv_et_al_2025_AI-simulated_entrepreneurship...,application/pdf,https://drive.google.com/file/d/1gwqTSpfg6Lqfb...,2025-11-28T01:55:47.932Z,2025-11-28T01:09:42.000Z,af68ab9652d883568dc3cea5c807c95c,1877980
7,1rqvhybG1D2BcaFuDEkV0FnjMo2Lhru-8,"Soleimani Dahaj, Arash & Cozzarin, Brian Paul ...",application/pdf,https://drive.google.com/file/d/1rqvhybG1D2Bca...,2025-11-25T05:45:13.752Z,2025-11-19T00:35:39.000Z,0843225079e3a58cb7f5b3b58f75a4d2,588709
8,1Z8EBN7S3qjnD9Uz7Hf5LP22pKh4UG0N6,"Cumming, D. J., Grilli, L., & Murtinu, S. (201...",application/pdf,https://drive.google.com/file/d/1Z8EBN7S3qjnD9...,2025-11-19T02:59:06.789Z,2025-11-19T00:34:51.000Z,9b90b667b4c8201b2d5965a269471b55,407891
9,1xt_TXB6csFjG-2BwWBGka-NYWO0scQYk,"Alperovych, Y., Manigart, S., Quas, A., & Stan...",application/pdf,https://drive.google.com/file/d/1xt_TXB6csFjG-...,2025-11-19T02:48:21.393Z,2025-11-19T00:34:17.000Z,09b4d6b8a7348e6ae4d05b1bd5e63089,949691



🔎 Pre-flight duplicate preview (first 10 targets):
- doi:10.1002/sej.1515 | filename='Li, Y. (2024). Venture capital exit after venture IPO. Strategic Entrepreneurshi...' | prefilter_hits=0 | sample=[]
- doi:10.1111/1467-8551.12803 | filename='Basnet, A. (2024). Post‐IPO lead venture capital firm involvement, merger‐relate...' | prefilter_hits=0 | sample=[]
- doi:10.1111/jfir.12412 | filename='Basnet, A. (2024). VC ownership post‐IPO- When, why, and how do VCs exit?. The J...' | prefilter_hits=0 | sample=[]
- url:https://arxiv.org/abs/2601.00810 | filename='Rashidi, M. (2025). Can Large Language Models Improve Venture Capital Exit Timin...' | prefilter_hits=0 | sample=[]
- doi:10.30574/wjarr.2024.22.1.1047 | filename='Soyombo, D. (2024). The role of policy and regulation in promoting green buildin...' | prefilter_hits=0 | sample=[]
- doi:10.1007/s43441-025-00773-3 | filename='Schulthess, D. (2025). The Inflation Reduction Act’s Impact Upon Early-Stage Ven...' | prefilter_hits=0 | samp

In [30]:
# ============================================================
# Cell 10: Notion upsert (create or update) logic (revised)
#  - LOCAL PDF → text extraction → LLM fields → Notion upsert
# ============================================================
#
# What this cell does:
# 1) Reads your Notion DB schema (self-healing: only writes properties that exist)
# 2) Finds an existing page (idempotency) by Paper ID / DOI / Landing URL when available
# 3) Extracts text from the LOCAL PDF (the file you downloaded in Cell 11 via fetch_res["local_path"])
# 4) Uses ChatGPT/OpenAI to generate "Core Idea / Methods / Findings / ..." from the extracted text
# 5) Creates or updates the Notion page
#
# Requirements:
# - NOTION_HEADERS, NOTION_DB_ID are set (Cell 02/03)
# - OPENAI_API_KEY is set (Cell 02)
# - openai_client is available OR we'll instantiate it
# - In Cell 11, call: notion_upsert_paper(row, drive_link=..., local_pdf_path=fetch_res["local_path"])
#
# Optional:
# - Install PDF text libs (recommended):
#   !pip -q install pymupdf pdfplumber
#
# Notes:
# - This does not do OCR. For scanned PDFs (image-only), text may be empty.
# - If your Notion DB has different property names, the self-healing mapper will adapt.
#

from __future__ import annotations

import os
import re
import json
import time
import pathlib
from typing import Any, Optional, Dict, List, Tuple

import requests
import pandas as pd

NOTION_TIMEOUT_SEC = float(os.getenv("NOTION_TIMEOUT_SEC", "30"))
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
OPENAI_TEMPERATURE = float(os.getenv("OPENAI_TEMPERATURE", "0.2"))

# Turn on enrichment from PDF text
NOTION_LLM_ENRICH = os.getenv("NOTION_LLM_ENRICH", "true").lower() in ("1", "true", "yes")

# PDF text extraction controls
PDF_TEXT_MAX_CHARS = int(os.getenv("PDF_TEXT_MAX_CHARS", "120000"))
PDF_TEXT_MAX_PAGES = int(os.getenv("PDF_TEXT_MAX_PAGES", "20"))

# Prefer matching in this order; keys are canonical (not Notion property names)
PREFERRED_MATCH_KEYS = [
    x.strip()
    for x in os.getenv("PREFERRED_MATCH_KEYS", "paper_id,doi,landing_url").split(",")
    if x.strip()
]

print("=== Notion Upsert Config (Cell 10) ===")
print("NOTION_DB_ID:", NOTION_DB_ID)
print("NOTION_TIMEOUT_SEC:", NOTION_TIMEOUT_SEC)
print("NOTION_LLM_ENRICH:", NOTION_LLM_ENRICH)
print("OPENAI_MODEL:", OPENAI_MODEL)
print("PDF_TEXT_MAX_PAGES:", PDF_TEXT_MAX_PAGES)
print("PDF_TEXT_MAX_CHARS:", PDF_TEXT_MAX_CHARS)
print("PREFERRED_MATCH_KEYS:", PREFERRED_MATCH_KEYS)
print("======================================\n")


# ------------------------------------------------------------
# OpenAI helper (Responses API preferred) - your shared logic
# ------------------------------------------------------------
def _openai_text(openai_client, model: str, system: str, user: str, temperature: float = 0.2) -> str:
    """
    Robust helper for OpenAI Python SDK.
    - Prefer Responses API
    - Extract text even when output_text is empty (SDK variants)
    """
    if hasattr(openai_client, "responses"):
        r = openai_client.responses.create(
            model=model,
            temperature=temperature,
            input=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        )

        out = (getattr(r, "output_text", None) or "").strip()
        if out:
            return out

        try:
            chunks = []
            for item in getattr(r, "output", []) or []:
                for c in getattr(item, "content", []) or []:
                    if isinstance(c, dict):
                        if c.get("type") == "output_text" and "text" in c:
                            chunks.append(c["text"])
                    else:
                        if getattr(c, "type", None) == "output_text" and getattr(c, "text", None):
                            chunks.append(c.text)
            out2 = "\n".join(chunks).strip()
            return out2
        except Exception:
            pass

        return str(r)

    if hasattr(openai_client, "chat") and hasattr(openai_client.chat, "completions"):
        r = openai_client.chat.completions.create(
            model=model,
            temperature=temperature,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        )
        return (r.choices[0].message.content or "").strip()

    raise AttributeError("openai_client does not support responses or chat.completions.")


def parse_json_object_loose(text: str) -> dict:
    """
    Extract the first JSON object from a string.
    """
    if not text or not text.strip():
        raise ValueError("Empty response text (cannot parse JSON).")

    try:
        return json.loads(text)
    except Exception:
        pass

    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        raise ValueError("No JSON object found in response text.")

    return json.loads(m.group(0))


# ------------------------------------------------------------
# PDF text extraction (LOCAL FILE)
# ------------------------------------------------------------
def _try_import_pymupdf():
    try:
        import fitz  # PyMuPDF
        return fitz
    except Exception:
        return None

def _try_import_pdfplumber():
    try:
        import pdfplumber
        return pdfplumber
    except Exception:
        return None

def _clean_text(s: str) -> str:
    s = s or ""
    s = s.replace("\x00", " ")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def extract_pdf_text_local(
    pdf_path: str,
    max_chars: int = PDF_TEXT_MAX_CHARS,
    max_pages: int = PDF_TEXT_MAX_PAGES,
) -> Dict[str, Any]:
    """
    Extract text from a local PDF.
    Returns:
      ok, text, method, pages_extracted, char_count, reason
    """
    p = pathlib.Path(pdf_path)
    if not pdf_path or not p.exists():
        return {"ok": False, "text": "", "method": None, "pages_extracted": 0, "char_count": 0, "reason": "pdf_not_found"}

    fitz = _try_import_pymupdf()
    if fitz:
        try:
            doc = fitz.open(str(p))
            chunks = []
            n_pages = min(len(doc), max_pages)
            for i in range(n_pages):
                chunks.append(doc.load_page(i).get_text("text") or "")
                if sum(len(c) for c in chunks) >= max_chars:
                    break
            text = _clean_text("\n".join(chunks))[:max_chars]
            return {
                "ok": bool(text.strip()),
                "text": text,
                "method": "pymupdf",
                "pages_extracted": n_pages,
                "char_count": len(text),
                "reason": None if text.strip() else "empty_text",
            }
        except Exception:
            pass  # fallback to pdfplumber

    pdfplumber = _try_import_pdfplumber()
    if pdfplumber:
        try:
            chunks = []
            with pdfplumber.open(str(p)) as pdf:
                n_pages = min(len(pdf.pages), max_pages)
                for i in range(n_pages):
                    chunks.append(pdf.pages[i].extract_text() or "")
                    if sum(len(c) for c in chunks) >= max_chars:
                        break
            text = _clean_text("\n".join(chunks))[:max_chars]
            return {
                "ok": bool(text.strip()),
                "text": text,
                "method": "pdfplumber",
                "pages_extracted": n_pages,
                "char_count": len(text),
                "reason": None if text.strip() else "empty_text",
            }
        except Exception as e:
            return {"ok": False, "text": "", "method": "pdfplumber", "pages_extracted": 0, "char_count": 0,
                    "reason": f"{type(e).__name__}: {str(e)[:160]}"}

    return {
        "ok": False,
        "text": "",
        "method": None,
        "pages_extracted": 0,
        "char_count": 0,
        "reason": "no_pdf_text_lib_available (install pymupdf or pdfplumber)",
    }


# ------------------------------------------------------------
# Notion API helpers
# ------------------------------------------------------------
def notion_get_database_schema(db_id: str) -> Dict[str, Any]:
    url = f"https://api.notion.com/v1/databases/{db_id}"
    r = requests.get(url, headers=NOTION_HEADERS, timeout=NOTION_TIMEOUT_SEC)
    if r.status_code != 200:
        raise RuntimeError(f"Notion DB read failed: {r.status_code} {r.text[:200]}")
    return r.json()

def notion_query_database(db_id: str, filter_obj: dict) -> dict:
    url = f"https://api.notion.com/v1/databases/{db_id}/query"
    payload = {"filter": filter_obj, "page_size": 10}
    r = requests.post(url, headers=NOTION_HEADERS, json=payload, timeout=NOTION_TIMEOUT_SEC)
    if r.status_code != 200:
        raise RuntimeError(f"Notion query failed: {r.status_code} {r.text[:200]}")
    return r.json()

def notion_create_page(db_id: str, properties: dict) -> dict:
    url = "https://api.notion.com/v1/pages"
    payload = {"parent": {"database_id": db_id}, "properties": properties}
    r = requests.post(url, headers=NOTION_HEADERS, json=payload, timeout=NOTION_TIMEOUT_SEC)
    if r.status_code != 200:
        raise RuntimeError(f"Notion create failed: {r.status_code} {r.text[:200]}")
    return r.json()

def notion_update_page(page_id: str, properties: dict) -> dict:
    url = f"https://api.notion.com/v1/pages/{page_id}"
    payload = {"properties": properties}
    r = requests.patch(url, headers=NOTION_HEADERS, json=payload, timeout=NOTION_TIMEOUT_SEC)
    if r.status_code != 200:
        raise RuntimeError(f"Notion update failed: {r.status_code} {r.text[:200]}")
    return r.json()


# ------------------------------------------------------------
# Mapping helpers (Notion property payloads)
# ------------------------------------------------------------
def _to_text(x: Any) -> str:
    """
    Convert non-string JSON values to a readable string.
    - list -> bullet lines
    - dict -> JSON string
    - None -> ""
    """
    if x is None:
        return ""
    if isinstance(x, str):
        return x
    if isinstance(x, list):
        items = []
        for v in x:
            if v is None:
                continue
            items.append(str(v).strip())
        items = [i for i in items if i]
        if not items:
            return ""
        # render as bullets (Japanese fields often look good this way)
        return "\n".join([f"- {i}" for i in items])
    if isinstance(x, dict):
        return json.dumps(x, ensure_ascii=False)
    return str(x)

def rt(s: Any):
    s2 = _to_text(s).strip()
    return {"rich_text": [{"type": "text", "text": {"content": s2}}]} if s2 else {"rich_text": []}

def title_prop(s: Any):
    s2 = _to_text(s).strip() or "Untitled Paper"
    return {"title": [{"type": "text", "text": {"content": s2}}]}


def ms(options):
    options = options or []
    cleaned = [{"name": str(x).strip()} for x in options if str(x).strip()]
    return {"multi_select": cleaned}

def url_prop(s: Optional[str]):
    s = (s or "").strip()
    return {"url": s if s else None}


# ------------------------------------------------------------
# Self-healing: discover actual DB properties and resolve mapping
# ------------------------------------------------------------
db_schema = notion_get_database_schema(NOTION_DB_ID)
DB_PROPERTIES = db_schema.get("properties", {}) or {}
DB_PROP_NAMES = set(DB_PROPERTIES.keys())

print("✅ Notion DB properties detected:", len(DB_PROP_NAMES))
print("   Sample:", sorted(list(DB_PROP_NAMES))[:20])

PROPERTY_NAME_CANDIDATES = {
    "Name": ["Name", "Title", "Paper Title"],
    "Authors & Year": ["Authors & Year", "Authors", "Author(s)", "Authors Year"],
    "Source": ["Source", "Venue", "Journal"],
    "Type": ["Type", "Paper Type"],
    "Core Idea": ["Core Idea", "Core", "Summary"],
    "Datasets": ["Datasets", "Dataset"],
    "Methods": ["Methods", "Method"],
    "Findings": ["Findings", "Key Findings"],
    "Notes": ["Notes", "Note", "Memo"],
    "Tags": ["Tags", "Tag", "Keywords"],
    "PDF Link": ["PDF Link", "PDF", "PDF URL", "Drive Link"],
    "Paper ID": ["Paper ID", "paper_id", "ID", "Key", "PaperId"],
    "DOI": ["DOI", "Doi"],
    "Landing URL": ["Landing URL", "URL", "Landing Page", "Link"],
}

def _pick_prop(candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in DB_PROP_NAMES:
            return c
    return None

RESOLVED = {k: _pick_prop(v) for k, v in PROPERTY_NAME_CANDIDATES.items()}

print("\n=== Resolved Notion property mapping ===")
for k, v in RESOLVED.items():
    if v:
        print(f"{k:12s} -> {v}")
print("======================================\n")


def build_props_safe(fields: dict) -> dict:
    """
    Build Notion properties dict, but only include properties that exist in DB.
    """
    props = {}

    if RESOLVED["Name"]:
        props[RESOLVED["Name"]] = title_prop(fields.get("name"))

    if RESOLVED["Authors & Year"]:
        props[RESOLVED["Authors & Year"]] = rt(fields.get("authors_year"))

    if RESOLVED["Source"]:
        props[RESOLVED["Source"]] = rt(fields.get("source"))

    if RESOLVED["Type"]:
        props[RESOLVED["Type"]] = rt(fields.get("type"))

    if RESOLVED["Core Idea"]:
        props[RESOLVED["Core Idea"]] = rt(fields.get("core_idea"))

    if RESOLVED["Datasets"]:
        props[RESOLVED["Datasets"]] = rt(fields.get("datasets"))

    if RESOLVED["Methods"]:
        props[RESOLVED["Methods"]] = rt(fields.get("methods"))

    if RESOLVED["Findings"]:
        props[RESOLVED["Findings"]] = rt(fields.get("findings"))

    if RESOLVED["Notes"]:
        props[RESOLVED["Notes"]] = rt(fields.get("notes"))

    if RESOLVED["Tags"]:
        props[RESOLVED["Tags"]] = ms(fields.get("tags", []))

    if RESOLVED["Paper ID"]:
        props[RESOLVED["Paper ID"]] = rt(fields.get("paper_id"))

    if RESOLVED["DOI"]:
        props[RESOLVED["DOI"]] = rt(fields.get("doi"))

    if RESOLVED["Landing URL"]:
        props[RESOLVED["Landing URL"]] = url_prop(fields.get("landing_url"))

    if RESOLVED["PDF Link"]:
        props[RESOLVED["PDF Link"]] = url_prop(fields.get("pdf_link"))

    return props


# ------------------------------------------------------------
# Idempotency: find existing page
# ------------------------------------------------------------
def _filter_rich_text_contains(prop_name: str, val: str) -> dict:
    return {"property": prop_name, "rich_text": {"contains": val}}

def _filter_url_equals(prop_name: str, val: str) -> dict:
    return {"property": prop_name, "url": {"equals": val}}

def find_existing_page_id(fields: dict) -> Optional[str]:
    checks: List[Tuple[str, dict]] = []

    if "paper_id" in PREFERRED_MATCH_KEYS and RESOLVED["Paper ID"]:
        v = (fields.get("paper_id") or "").strip()
        if v:
            checks.append(("paper_id", _filter_rich_text_contains(RESOLVED["Paper ID"], v)))

    if "doi" in PREFERRED_MATCH_KEYS and RESOLVED["DOI"]:
        v = (fields.get("doi") or "").strip()
        if v and v.lower() not in ("none", "nan"):
            checks.append(("doi", _filter_rich_text_contains(RESOLVED["DOI"], v)))

    if "landing_url" in PREFERRED_MATCH_KEYS and RESOLVED["Landing URL"]:
        v = (fields.get("landing_url") or "").strip()
        if v:
            checks.append(("landing_url", _filter_url_equals(RESOLVED["Landing URL"], v)))

    for _, fobj in checks:
        res = notion_query_database(NOTION_DB_ID, fobj)
        results = res.get("results", [])
        if results:
            return results[0]["id"]

    return None


# ------------------------------------------------------------
# Canonical field builder from a targets_df row
# ------------------------------------------------------------
def build_canonical_fields(row: pd.Series, drive_link: Optional[str] = None) -> dict:
    year = row.get("year")
    year_str = str(int(year)) if pd.notna(year) else "n.d."
    authors = str(row.get("authors") or "").strip()
    authors_year = f"{authors} ({year_str})" if authors else f"(Unknown) ({year_str})"

    return {
        # Human-readable fields
        "name": str(row.get("title") or "Untitled Paper").strip(),
        "authors_year": authors_year,
        "source": str(row.get("venue") or "").strip(),
        # LLM fill targets
        "type": "",
        "core_idea": "",
        "datasets": "",
        "methods": "",
        "findings": "",
        "notes": "",
        "tags": [],
        # IDs/links
        "paper_id": str(row.get("paper_id") or "").strip(),
        "doi": str(row.get("doi") or "").strip(),
        "landing_url": str(row.get("landing_url") or "").strip(),
        "pdf_link": drive_link,
    }


# ------------------------------------------------------------
# LLM: generate notion fields from PDF text
# ------------------------------------------------------------
def llm_generate_fields_from_pdf_text(fields_seed: dict, paper_text: str) -> dict:
    """
    Returns JSON with keys:
      name, authors_year, source, type, core_idea, datasets, methods, findings, notes, tags
    """
    if "openai_client" not in globals():
        from openai import OpenAI
        openai_client = OpenAI(api_key=OPENAI_API_KEY)
    else:
        openai_client = globals()["openai_client"]

    system = "You are a precise research assistant who writes concise database-ready summaries."

    user = f"""
Create a Literature Database entry from the paper text below.

Output language rules:
- Name (title): English (single line)
- Authors & Year: English (format: "Last, First (Year)" or keep original order if unclear)
- Source: English (journal / venue)
- Type: English (short noun phrase, e.g. "Empirical", "Policy Evaluation")
- Tags: English (JSON array of short tags, 2–6 items, Title Case)

- Core Idea: Japanese (2–4 sentences)
- Datasets: Japanese (箇条書き。分からなければ「不明」)
- Methods: Japanese (箇条書き。分からなければ「不明」)
- Findings: Japanese (箇条書き。分からなければ「不明」)
- Notes: Japanese (短く)

Return JSON ONLY with keys:
name, authors_year, source, type, core_idea, datasets, methods, findings, notes, tags

Hints (may be incomplete):
Title: {fields_seed.get("name","")}
Authors & Year: {fields_seed.get("authors_year","")}
Source: {fields_seed.get("source","")}

Paper text (first ~{PDF_TEXT_MAX_CHARS} chars, extracted from PDF):
{paper_text[:PDF_TEXT_MAX_CHARS]}
""".strip()

    out = _openai_text(openai_client, OPENAI_MODEL, system, user, temperature=OPENAI_TEMPERATURE)
    return parse_json_object_loose(out)


# ------------------------------------------------------------
# Public: upsert (create or update) using LOCAL PDF path
# ------------------------------------------------------------
def notion_upsert_paper(
    row: pd.Series,
    drive_link: Optional[str] = None,
    local_pdf_path: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Upsert a paper row into NOTION_DB_ID.
    - Uses local_pdf_path to extract text and fill Core Idea/Methods/Findings via LLM.
    - If a match is found -> update, else -> create.

    Returns:
      status: created | updated | failed
      page_id, page_url, reason, pdf_text_meta
    """
    pdf_text_meta = None
    try:
        fields = build_canonical_fields(row, drive_link=drive_link)

        # --- Extract text from LOCAL PDF ---
        paper_text = None
        if local_pdf_path:
            pdf_text_meta = extract_pdf_text_local(
                local_pdf_path,
                max_chars=PDF_TEXT_MAX_CHARS,
                max_pages=PDF_TEXT_MAX_PAGES,
            )
            if pdf_text_meta.get("ok"):
                paper_text = pdf_text_meta["text"]

        # --- LLM enrichment from PDF text ---
        if NOTION_LLM_ENRICH:
            if paper_text and paper_text.strip():
                enrich = llm_generate_fields_from_pdf_text(fields, paper_text)
                # Merge: LLM wins when present & non-empty
                for k in ["name","authors_year","source","type","core_idea","datasets","methods","findings","notes","tags"]:
                    if k not in enrich or enrich[k] is None:
                        continue
                
                    if k == "tags":
                        # tags is expected to be a list of strings
                        if isinstance(enrich[k], list) and len(enrich[k]) > 0:
                            fields[k] = enrich[k]
                        continue
                
                    # everything else: accept non-empty after stringification
                    if _to_text(enrich[k]).strip() != "":
                        fields[k] = enrich[k]
        
            else:
                # Fail-soft: keep metadata-only record but explain
                note = fields.get("notes", "")
                reason = (pdf_text_meta or {}).get("reason") if pdf_text_meta else "no_local_pdf_path_provided"
                fields["notes"] = (note + f"\n[PDF text unavailable] {reason}").strip()

        props = build_props_safe(fields)

        existing_id = find_existing_page_id(fields)
        if existing_id:
            page = notion_update_page(existing_id, props)
            return {"status": "updated", "page_id": page.get("id"), "page_url": page.get("url"), "reason": None, "pdf_text_meta": pdf_text_meta}
        else:
            page = notion_create_page(NOTION_DB_ID, props)
            return {"status": "created", "page_id": page.get("id"), "page_url": page.get("url"), "reason": None, "pdf_text_meta": pdf_text_meta}

    except Exception as e:
        return {"status": "failed", "page_id": None, "page_url": None, "reason": f"{type(e).__name__}: {str(e)[:220]}", "pdf_text_meta": pdf_text_meta}


print("✅ Cell 10 ready: notion_upsert_paper(row, drive_link=..., local_pdf_path=...)")
print("   - Extracts text from LOCAL PDF and uses LLM to fill Core Idea/Methods/Findings/etc (if DB has those props).")


=== Notion Upsert Config (Cell 10) ===
NOTION_DB_ID: 2a98e0e4d16280cbb6cbdcd1ebedee54
NOTION_TIMEOUT_SEC: 30.0
NOTION_LLM_ENRICH: True
OPENAI_MODEL: gpt-4.1-mini
PDF_TEXT_MAX_PAGES: 20
PDF_TEXT_MAX_CHARS: 120000
PREFERRED_MATCH_KEYS: ['paper_id', 'doi', 'landing_url']

✅ Notion DB properties detected: 13
   Sample: ['Authors & Year', 'Core Idea', 'Created time', 'Datasets', 'Findings', 'Methods', 'Name', 'Notes', 'PDF Link', 'Papers', 'Source', 'Tags', 'Type']

=== Resolved Notion property mapping ===
Name         -> Name
Authors & Year -> Authors & Year
Source       -> Source
Type         -> Type
Core Idea    -> Core Idea
Datasets     -> Datasets
Methods      -> Methods
Findings     -> Findings
Notes        -> Notes
Tags         -> Tags
PDF Link     -> PDF Link

✅ Cell 10 ready: notion_upsert_paper(row, drive_link=..., local_pdf_path=...)
   - Extracts text from LOCAL PDF and uses LLM to fill Core Idea/Methods/Findings/etc (if DB has those props).


In [31]:
# ============================================================
# Cell 11: Main processing loop (fetch → drive → notion) (revised)
# ============================================================
#
# This revision assumes:
# - Cell 06 provides: fetch_pdf_from_candidates(...)
# - Cell 09 provides: drive_list_pdfs_in_folder(...), drive_upload_pdf_with_dedupe(...)
# - Cell 10 provides: notion_upsert_paper(...)
# - targets_df includes: paper_id, title, year, authors, venue, doi, landing_url, pdf_url_candidates
#
# Improvements vs previous:
# - Uses the improved fetch attempts diagnostics (status_code/content_type/bytes/error)
# - Refreshes Drive listing periodically (optional) so dedupe stays accurate as we upload
# - Writes clear run logs (drive_results, notion_results, failures) + JSON attempts per paper
# - Continues to Notion even if Drive upload fails (so you can track failed downloads/uploads)
#

from __future__ import annotations

import os
import json
import time
import pathlib
from datetime import datetime
from typing import Any, Dict, List, Optional

import pandas as pd

# ------------------------------------------------------------
# Run config
# ------------------------------------------------------------
RUN_TS = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = pathlib.Path(os.getenv("RUN_DIR", f"./artifacts/019_runs/{RUN_TS}"))
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Optional: refresh Drive listing every N successful uploads
DRIVE_REFRESH_EVERY = int(os.getenv("DRIVE_REFRESH_EVERY", "10"))

# Optional: cap per run (extra safety)
HARD_MAX_PER_RUN = int(os.getenv("HARD_MAX_PER_RUN", "999999"))

print("=== Run Info ===")
print("RUN_TS:", RUN_TS)
print("RUN_DIR:", RUN_DIR)
print("Targets:", len(targets_df))
print("DRY_RUN:", DRY_RUN)
print("DRIVE_REFRESH_EVERY:", DRIVE_REFRESH_EVERY)
print("================\n")

# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------
def _now_iso() -> str:
    return datetime.utcnow().isoformat() + "Z"

def _safe_str(x: Any, n: int = 240) -> str:
    s = "" if x is None else str(x)
    return s[:n]

def _json_dumps(obj: Any) -> str:
    try:
        return json.dumps(obj, ensure_ascii=False)
    except Exception:
        return json.dumps(str(obj), ensure_ascii=False)

def _get_row_value(row: pd.Series, key: str, default=None):
    try:
        v = row.get(key, default)
        return v
    except Exception:
        return default


# ------------------------------------------------------------
# Preload Drive listing for dedupe (Cell 09)
# ------------------------------------------------------------
if "drive_files_df" not in globals():
    print("🔎 Listing existing PDFs in target Drive folder (for dedupe)...")
    drive_files_df = drive_list_pdfs_in_folder(drive_service, DRIVE_FOLDER_ID, max_items=DRIVE_LIST_MAX)
    print("✅ Drive PDFs listed:", len(drive_files_df))

uploads_since_refresh = 0

# ------------------------------------------------------------
# Logs (written at end)
# ------------------------------------------------------------
drive_logs: List[Dict[str, Any]] = []
notion_logs: List[Dict[str, Any]] = []
failure_logs: List[Dict[str, Any]] = []

# Optional: ensure we do not exceed HARD_MAX_PER_RUN
targets_iter = targets_df.head(min(len(targets_df), HARD_MAX_PER_RUN))

# ------------------------------------------------------------
# Main loop
# ------------------------------------------------------------
for i, (_, row) in enumerate(targets_iter.iterrows(), start=1):
    paper_id = str(_get_row_value(row, "paper_id", "") or "").strip()
    title = str(_get_row_value(row, "title", "") or "").strip()
    year = _get_row_value(row, "year", None)
    doi = _get_row_value(row, "doi", None)
    landing_url = _get_row_value(row, "landing_url", None)

    print(f"\n[{i}/{len(targets_iter)}] {paper_id} | {title[:90]}")

    # --------------------------------------------------------
    # 1) Fetch PDF (multi-candidate)
    # --------------------------------------------------------
    url_candidates = list(_get_row_value(row, "pdf_url_candidates", []) or [])

    # If you added a publisher-aware augmentation helper in Cell 07, you can use it here:
    if "augment_pdf_candidates_publisher_aware" in globals():
        url_candidates = augment_pdf_candidates_publisher_aware(row, url_candidates)

    try:
        fetch_res = fetch_pdf_from_candidates(
            paper_id=paper_id,
            url_candidates=url_candidates,
            out_dir=DOWNLOAD_DIR,
            filename_hint=None,     # optional: pass a pretty name if you want
            max_attempts=8,
        )
    except Exception as e:
        fetch_res = {
            "ok": False,
            "paper_id": paper_id,
            "chosen_url": None,
            "local_path": None,
            "bytes": 0,
            "sha256": None,
            "attempts": [],
            "final_error_code": "fetch_exception",
            "final_error_message": f"{type(e).__name__}: {_safe_str(e)}",
        }

    if not fetch_res.get("ok"):
        print(f"  ❌ Fetch failed: {fetch_res.get('final_error_code')} | {fetch_res.get('final_error_message')}")
        failure_logs.append({
            "ts": _now_iso(),
            "stage": "fetch",
            "paper_id": paper_id,
            "title": title,
            "error_code": fetch_res.get("final_error_code"),
            "error_message": _safe_str(fetch_res.get("final_error_message")),
            "attempts_json": _json_dumps(fetch_res.get("attempts", [])),
        })
        # No Drive/Notion if no local PDF
        continue

    print(f"  ✅ Fetch OK: bytes={fetch_res.get('bytes')} sha256={str(fetch_res.get('sha256'))[:12]}...")

    # --------------------------------------------------------
    # 2) Drive upload (with dedupe)
    # --------------------------------------------------------
    if DRY_RUN:
        drive_res = {
            "status": "dry_run",
            "paper_id": paper_id,
            "drive_file_id": None,
            "drive_link": None,
            "filename": None,
            "reason": "DRY_RUN enabled",
            "dedupe": {"method": "dry_run"},
            "local_sha256": fetch_res.get("sha256"),
        }
        print("  🧪 DRY_RUN: skipping Drive upload.")
    else:
        try:
            drive_res = drive_upload_pdf_with_dedupe(
                service=drive_service,
                folder_id=DRIVE_FOLDER_ID,
                local_path=fetch_res.get("local_path"),
                paper_row=row,
                paper_id=paper_id,
                local_sha256=fetch_res.get("sha256"),
                drive_df=drive_files_df,
                llm_enabled=LLM_DUPLICATE_ENABLED,
                llm_confidence_threshold=0.85,
            )
        except Exception as e:
            drive_res = {
                "status": "failed",
                "paper_id": paper_id,
                "drive_file_id": None,
                "drive_link": None,
                "filename": None,
                "reason": f"{type(e).__name__}: {_safe_str(e)}",
                "dedupe": {"method": "exception"},
                "local_sha256": fetch_res.get("sha256"),
            }

        if drive_res.get("status") == "uploaded":
            print(f"  ✅ Drive uploaded: {drive_res.get('filename')}")
            uploads_since_refresh += 1
        elif drive_res.get("status") == "skipped_duplicate":
            print(f"  ♻️ Drive duplicate skipped: {drive_res.get('filename')}")
        else:
            print(f"  ❌ Drive failed: {drive_res.get('reason')}")

    # Optional: refresh Drive list so dedupe stays accurate as we upload
    if (not DRY_RUN) and DRIVE_REFRESH_EVERY > 0 and uploads_since_refresh >= DRIVE_REFRESH_EVERY:
        print("  🔄 Refreshing Drive listing for dedupe...")
        drive_files_df = drive_list_pdfs_in_folder(drive_service, DRIVE_FOLDER_ID, max_items=DRIVE_LIST_MAX)
        uploads_since_refresh = 0
        print("  ✅ Refreshed Drive listing:", len(drive_files_df))

    drive_logs.append({
        "ts": _now_iso(),
        "paper_id": paper_id,
        "title": title,
        "year": year,
        "doi": doi,
        "landing_url": landing_url,
        "chosen_url": fetch_res.get("chosen_url"),
        "download_bytes": fetch_res.get("bytes"),
        "sha256": fetch_res.get("sha256"),
        "attempts_json": _json_dumps(fetch_res.get("attempts", [])),
        "drive_status": drive_res.get("status"),
        "drive_file_id": drive_res.get("drive_file_id"),
        "drive_link": drive_res.get("drive_link"),
        "drive_filename": drive_res.get("filename"),
        "drive_reason": drive_res.get("reason"),
        "dedupe_method": (drive_res.get("dedupe") or {}).get("method"),
        "dedupe_confidence": (drive_res.get("dedupe") or {}).get("confidence"),
        "dedupe_matched_file_id": (drive_res.get("dedupe") or {}).get("matched_file_id"),
    })

    if drive_res.get("status") == "failed":
        failure_logs.append({
            "ts": _now_iso(),
            "stage": "drive",
            "paper_id": paper_id,
            "title": title,
            "error_code": "drive_failed",
            "error_message": _safe_str(drive_res.get("reason")),
            "attempts_json": _json_dumps(fetch_res.get("attempts", [])),
        })
        # Continue to Notion anyway (tracking record)
        # (If you want to skip Notion on drive failure, replace with: continue)

    # --------------------------------------------------------
    # 3) Notion upsert (Cell 10 revised)
    # --------------------------------------------------------
    if DRY_RUN:
        notion_res = {"status": "dry_run", "page_id": None, "page_url": None, "reason": "DRY_RUN enabled"}
        print("  🧪 DRY_RUN: skipping Notion upsert.")
    else:
        notion_res = notion_upsert_paper(
            row=row,
            drive_link=drive_res.get("drive_link"),
            local_pdf_path=fetch_res.get("local_path"), 
        )

        if notion_res.get("status") in ("created", "updated"):
            print(f"  ✅ Notion {notion_res.get('status')}: {notion_res.get('page_id')}")
        else:
            print(f"  ❌ Notion failed: {notion_res.get('reason')}")

    notion_logs.append({
        "ts": _now_iso(),
        "paper_id": paper_id,
        "title": title,
        "notion_status": notion_res.get("status"),
        "notion_page_id": notion_res.get("page_id"),
        "notion_page_url": notion_res.get("page_url"),
        "notion_reason": notion_res.get("reason"),
        "drive_link": drive_res.get("drive_link"),
    })

    if notion_res.get("status") == "failed":
        failure_logs.append({
            "ts": _now_iso(),
            "stage": "notion",
            "paper_id": paper_id,
            "title": title,
            "error_code": "notion_failed",
            "error_message": _safe_str(notion_res.get("reason")),
            "attempts_json": _json_dumps(fetch_res.get("attempts", [])),
        })

# ------------------------------------------------------------
# Persist run logs
# ------------------------------------------------------------
drive_results_df = pd.DataFrame(drive_logs)
notion_results_df = pd.DataFrame(notion_logs)
failures_df = pd.DataFrame(failure_logs)

drive_csv = RUN_DIR / "drive_results.csv"
notion_csv = RUN_DIR / "notion_results.csv"
failures_csv = RUN_DIR / "failures.csv"

drive_results_df.to_csv(drive_csv, index=False)
notion_results_df.to_csv(notion_csv, index=False)
failures_df.to_csv(failures_csv, index=False)

print("\n=== Run Outputs ===")
print("Drive results:", drive_csv)
print("Notion results:", notion_csv)
print("Failures:", failures_csv)

display(drive_results_df.head(10))
display(notion_results_df.head(10))
display(failures_df.head(10))

# End of Cell 11


=== Run Info ===
RUN_TS: 20260112T044552Z
RUN_DIR: artifacts/019_runs/20260112T044552Z
Targets: 50
DRY_RUN: False
DRIVE_REFRESH_EVERY: 10


[1/50] doi:10.1002/sej.1515 | Venture capital exit after venture IPO
  ❌ Fetch failed: paywall_suspected | http_status_403

[2/50] doi:10.1111/1467-8551.12803 | Post‐IPO lead venture capital firm involvement, merger‐related litigation and target firm 
  ❌ Fetch failed: paywall_suspected | http_status_403

[3/50] doi:10.1111/jfir.12412 | VC ownership post‐IPO: When, why, and how do VCs exit?
  ❌ Fetch failed: paywall_suspected | http_status_403

[4/50] url:https://arxiv.org/abs/2601.00810 | Can Large Language Models Improve Venture Capital Exit Timing After IPO?
  ✅ Fetch OK: bytes=203687 sha256=4a589f23c324...
  ✅ Drive uploaded: Rashidi, M. (2025). Can Large Language Models Improve Venture Capital Exit Timing After IPO?. arXiv (Cornell University).pdf
  ✅ Notion created: 2e68e0e4-d162-8147-bd42-e4d5b451e23e

[5/50] doi:10.30574/wjarr.2024.22.1.104

,ts,paper_id,title,year,doi,landing_url,chosen_url,download_bytes,sha256,attempts_json,drive_status,drive_file_id,drive_link,drive_filename,drive_reason,dedupe_method,dedupe_confidence,dedupe_matched_file_id
0,2026-01-12T04:46:04.922518Z,url:https://arxiv.org/abs/2601.00810,Can Large Language Models Improve Venture Capi...,2025,NaN,https://arxiv.org/abs/2601.00810,https://arxiv.org/pdf/2601.00810,203687,4a589f23c324df14c8856557bb0d3e2e116f799ac6c2af...,"[{""url"": ""https://arxiv.org/pdf/2601.00810"", ""...",uploaded,1t5c5Q3vuRfIU1xEsne2U8eTUte7wm4X8,https://drive.google.com/file/d/1t5c5Q3vuRfIU1...,"Rashidi, M. (2025). Can Large Language Models ...",None,none,NaN,None
1,2026-01-12T04:46:22.300792Z,doi:10.30574/wjarr.2024.22.1.1047,The role of policy and regulation in promoting...,2024,10.30574/wjarr.2024.22.1.1047,https://doi.org/10.30574/wjarr.2024.22.1.1047,https://wjarr.com/sites/default/files/WJARR-20...,848914,cc3f74a9f0f3fcbc465c9a372542914a35d0e2bb6b199b...,"[{""url"": ""https://wjarr.com/sites/default/file...",uploaded,1KTxFifWgy6cZyqazNw1pVg6D1pC-HU2y,https://drive.google.com/file/d/1KTxFifWgy6cZy...,"Soyombo, D. (2024). The role of policy and reg...",None,none,NaN,None
2,2026-01-12T04:46:37.299158Z,doi:10.1007/s43441-025-00773-3,The Inflation Reduction Act’s Impact Upon Earl...,2025,10.1007/s43441-025-00773-3,https://doi.org/10.1007/s43441-025-00773-3,https://link.springer.com/content/pdf/10.1007/...,1555137,65ab52d1eac6f1b15a94bf12d46e9730a3e5e2037ef248...,"[{""url"": ""https://link.springer.com/content/pd...",uploaded,1kAiHwosZOPqFHA2E3yMox_opzinhSA72,https://drive.google.com/file/d/1kAiHwosZOPqFH...,"Schulthess, D. (2025). The Inflation Reduction...",None,none,NaN,None
3,2026-01-12T04:47:01.050506Z,doi:10.48550/arxiv.2307.03718,Frontier AI Regulation: Managing Emerging Risk...,2023,10.48550/arxiv.2307.03718,https://arxiv.org/abs/2307.03718,https://arxiv.org/pdf/2307.03718,1509008,42dc01e5ca84a06631a1fc8b3698ea5bee0574e0d923cf...,"[{""url"": ""https://arxiv.org/pdf/2307.03718"", ""...",uploaded,1zfSCmBxPOxceYFpqnS8FDDbDMvmsliPW,https://drive.google.com/file/d/1zfSCmBxPOxceY...,"Anderljung, M. (2023). Frontier AI Regulation-...",None,none,NaN,None
4,2026-01-12T04:47:23.504634Z,doi:10.36948/ijfmr.2025.v07i03.46824,UNDERSTANDING THE ROLE OF VENTURE CAPITAL BACK...,2025,10.36948/ijfmr.2025.v07i03.46824,https://doi.org/10.36948/ijfmr.2025.v07i03.46824,https://www.ijfmr.com/papers/2025/3/46824.pdf,253575,34abae06b8e975b5fa9c9caff27e15e13c8278b9ce57e0...,"[{""url"": ""https://www.ijfmr.com/papers/2025/3/...",uploaded,1Q50qFgwGu5GQ3EYJkXHh26CkX2PrOxXU,https://drive.google.com/file/d/1Q50qFgwGu5GQ3...,"S, V. (2025). UNDERSTANDING THE ROLE OF VENTUR...",None,none,NaN,None
5,2026-01-12T04:47:38.973440Z,doi:10.48550/arxiv.2509.14448,VCBench: Benchmarking LLMs in Venture Capital,2025,10.48550/arxiv.2509.14448,https://arxiv.org/abs/2509.14448,https://arxiv.org/pdf/2509.14448,841826,8624da90273c822c34e5c3da9523e644566b37e1fc7f80...,"[{""url"": ""https://arxiv.org/pdf/2509.14448"", ""...",uploaded,1mITkh-Z3UaOtnWJ2_eayp-8xVyaLg52E,https://drive.google.com/file/d/1mITkh-Z3UaOtn...,"Chen, R. (2025). VCBench- Benchmarking LLMs in...",None,none,NaN,None
6,2026-01-12T04:48:02.890643Z,url:https://arxiv.org/abs/2509.14448v1,VCBench: Benchmarking LLMs in Venture Capital,2025,NaN,https://arxiv.org/abs/2509.14448v1,https://arxiv.org/pdf/2509.14448v1,841826,8624da90273c822c34e5c3da9523e644566b37e1fc7f80...,"[{""url"": ""https://arxiv.org/pdf/2509.14448v1"",...",skipped_duplicate,1mITkh-Z3UaOtnWJ2_eayp-8xVyaLg52E,https://drive.google.com/file/d/1mITkh-Z3UaOtn...,"Chen, R. (2025). VCBench- Benchmarking LLMs in...",Duplicate detected by LLM (confidence=0.95): T...,llm,0.95,1mITkh-Z3UaOtnWJ2_eayp-8xVyaLg52E
7,2026-01-12T04:48:22.509667Z,doi:10.3389/frai.2023.1014317,Learning private equity recommitment strategie...,2023,10.3389/frai.2023.1014317,https://doi.org/10.3389/frai.2023.1014317,https://www.frontiersin.org/articles/10.3389/f...,2058674,

,ts,paper_id,title,notion_status,notion_page_id,notion_page_url,notion_reason,drive_link
0,2026-01-12T04:46:18.041264Z,url:https://arxiv.org/abs/2601.00810,Can Large Language Models Improve Venture Capi...,created,2e68e0e4-d162-8147-bd42-e4d5b451e23e,https://www.notion.so/Can-Large-Language-Model...,None,https://drive.google.com/file/d/1t5c5Q3vuRfIU1...
1,2026-01-12T04:46:31.599861Z,doi:10.30574/wjarr.2024.22.1.1047,The role of policy and regulation in promoting...,created,2e68e0e4-d162-818b-bb66-e668d0b1c2b7,https://www.notion.so/The-Role-of-Policy-and-R...,None,https://drive.google.com/file/d/1KTxFifWgy6cZy...
2,2026-01-12T04:46:55.088612Z,doi:10.1007/s43441-025-00773-3,The Inflation Reduction Act’s Impact Upon Earl...,created,2e68e0e4-d162-81dc-842f-cd2e1e03f345,https://www.notion.so/The-Inflation-Reduction-...,None,https://drive.google.com/file/d/1kAiHwosZOPqFH...
3,2026-01-12T04:47:15.548752Z,doi:10.48550/arxiv.2307.03718,Frontier AI Regulation: Managing Emerging Risk...,created,2e68e0e4-d162-81c4-aa9e-c8579712b083,https://www.notion.so/Frontier-AI-Regulation-M...,None,https://drive.google.com/file/d/1zfSCmBxPOxceY...
4,2026-01-12T04:47:36.774924Z,doi:10.36948/ijfmr.2025.v07i03.46824,UNDERSTANDING THE ROLE OF VENTURE CAPITAL BACK...,created,2e68e0e4-d162-8193-a96c-ea8098ce7922,https://www.notion.so/Understanding-the-Role-o...,None,https://drive.google.com/file/d/1Q50qFgwGu5GQ3...
5,2026-01-12T04:47:53.231503Z,doi:10.48550/arxiv.2509.14448,VCBench: Benchmarking LLMs in Venture Capital,created,2e68e0e4-d162-81e1-a8f8-dd409815fd15,https://www.notion.so/VCBench-Benchmarking-LLM...,None,https://drive.google.com/file/d/1mITkh-Z3UaOtn...
6,2026-01-12T04:48:17.910178Z,url:https://arxiv.org/abs/2509.14448v1,VCBench: Benchmarking LLMs in Venture Capital,created,2e68e0e4-d162-818a-857b-d662a761207b,https://www.notion.so/VCBench-Benchmarking-LLM...,None,https://drive.google.com/file/d/1mITkh-Z3UaOtn...
7,2026-01-12T04:48:37.161171Z,doi:10.3389/frai.2023.1014317,Learning private equity recommitment strategie...,created,2e68e0e4-d162-817b-ac6b-faaaadf6f919,https://www.notion.so/Learning-private-equity-...,None,https://drive.google.com/file/d/1c08_OnqaYs7EM...
8,2026-01-12T04:49:24.518817Z,doi:10.1007/s44163-024-00109-4,Managing the race to the moon: Global policy a...,created,2e68e0e4-d162-81fd-852b-e2bc023942d0,https://www.notion.so/Managing-the-Race-to-the...,None,https://drive.google.com/file/d/1rZnpNr_V46pi9...
9,2026-01-12T04:49:45.168555Z,doi:10.3897/imafungus.16.144989,Symbiotic synergy: How Arbuscular Mycorrhizal ...,created,2e68e0e4-d162-81b3-80fa-e4471be309ca,https://www.notion.so/Symbiotic-synergy-How-Ar...,None,https://drive.google.com/file/d/1a8Vg4ThFKD5Cw...


,ts,stage,paper_id,title,error_code,error_message,attempts_json
0,2026-01-12T04:45:55.864846Z,fetch,doi:10.1002/sej.1515,Venture capital exit after venture IPO,paywall_suspected,http_status_403,"[{""url"": ""https://onlinelibrary.wiley.com/doi/..."
1,2026-01-12T04:45:59.273583Z,fetch,doi:10.1111/1467-8551.12803,Post‐IPO lead venture capital firm involvement...,paywall_suspected,http_status_403,"[{""url"": ""https://onlinelibrary.wiley.com/doi/..."
2,2026-01-12T04:46:02.724598Z,fetch,doi:10.1111/jfir.12412,"VC ownership post‐IPO: When, why, and how do V...",paywall_suspected,http_status_403,"[{""url"": ""https://onlinelibrary.wiley.com/doi/..."
3,2026-01-12T04:46:58.709243Z,fetch,doi:10.1146/annurev-financial-111021-100657,IPOs and SPACs: Recent Developments,paywall_suspected,http_status_403,"[{""url"": ""https://www.annualreviews.org/doi/pd..."
4,2026-01-12T04:47:19.167129Z,fetch,doi:10.1093/rfs/hhad071,Common Venture Capital Investors and Startup G...,paywall_suspected,http_status_403,"[{""url"": ""https://academic.oup.com/rfs/advance..."
5,2026-01-12T04:47:56.954506Z,fetch,doi:10.3390/systems12030072,Overcoming Uncertainty in Novel Technologies: ...,paywall_suspected,http_status_403,"[{""url"": ""https://www.mdpi.com/2079-8954/12/3/..."
6,2026-01-12T04:48:00.565978Z,fetch,doi:10.3390/app151810060,Prioritizing Early-Stage Start-Up Investment A...,paywall_suspected,http_status_403,"[{""url"": ""https://www.mdpi.com/2076-3417/15/18..."
7,2026-01-12T04:48:40.641928Z,fetch,doi:10.3390/jrfm17040159,Pathways to Success: The Interplay of Industry...,paywall_suspected,http_status_403,"[{""url"": ""https://www.mdpi.com/1911-8074/17/4/..."
8,2026-01-12T04:48:44.097249Z,fetch,doi:10.1002/bse.3587,Is biodiversity disclosure emerging as a key t...,paywall_suspected,http_status_403,"[{""url"": ""https://onlinelibrary.wiley.com/doi/..."
9,2026-01-12T04:48:47.511200Z,fetch,doi:10.1002/anie.202425439,g‐C<sub>3</sub>N<sub>4</sub> S‐Scheme Homojunc...,paywall_suspected,http_status_403,"[{""url"": ""https://onlinelibrary.wiley.com/doi/..."


In [32]:
# ============================================================
# Cell 12: Aggregation of success / failure metrics
# ============================================================
#
# This cell aggregates the run outputs produced by Cell 11:
# - drive_results.csv
# - notion_results.csv
# - failures.csv
#
# It produces:
# - A compact run summary (counts + rates)
# - Breakdown tables (by stage / status / publisher-domain hints)
# - A "top failure reasons" view for quick debugging
# - Optional CSV exports of the aggregated metrics
#

from __future__ import annotations

import os
import json
import pathlib
from urllib.parse import urlparse

import pandas as pd

# -----------------------------
# Inputs (from Cell 11)
# -----------------------------
RUN_DIR = pathlib.Path(RUN_DIR)  # ensure Path
drive_csv = RUN_DIR / "drive_results.csv"
notion_csv = RUN_DIR / "notion_results.csv"
failures_csv = RUN_DIR / "failures.csv"

assert drive_csv.exists(), f"Missing: {drive_csv}"
assert notion_csv.exists(), f"Missing: {notion_csv}"
assert failures_csv.exists(), f"Missing: {failures_csv}"

# Optional export path
METRICS_DIR = RUN_DIR / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_METRICS = os.getenv("EXPORT_METRICS", "true").lower() in ("1", "true", "yes")

print("=== Metrics Inputs ===")
print("RUN_DIR:", RUN_DIR)
print("drive_results:", drive_csv)
print("notion_results:", notion_csv)
print("failures:", failures_csv)
print("EXPORT_METRICS:", EXPORT_METRICS)
print("======================\n")

# -----------------------------
# Load data
# -----------------------------
drive_df = pd.read_csv(drive_csv)
notion_df = pd.read_csv(notion_csv)
fail_df = pd.read_csv(failures_csv)

print("✅ Loaded:")
print(" - drive_df:", drive_df.shape)
print(" - notion_df:", notion_df.shape)
print(" - fail_df :", fail_df.shape)


# -----------------------------
# Helpers
# -----------------------------
def _domain(url: str) -> str:
    url = str(url or "").strip()
    if not url:
        return ""
    try:
        return urlparse(url).netloc.lower()
    except Exception:
        return ""

def _safe_str(x):
    return "" if pd.isna(x) else str(x)

def _value_counts(df: pd.DataFrame, col: str, topn: int = 20) -> pd.DataFrame:
    if col not in df.columns:
        return pd.DataFrame({"note": [f"Column not found: {col}"]})
    vc = df[col].fillna("").astype(str).value_counts().head(topn)
    return vc.reset_index().rename(columns={"index": col, col: "count"})


# -----------------------------
# Join (paper_id as key)
# -----------------------------
# drive_df rows typically exist only for fetch-ok items
# notion_df rows exist for items we attempted to upsert
# We'll build a unified "attempts" table to compute rates.
base_cols = ["paper_id", "title", "year", "doi", "landing_url", "chosen_url"]
drive_cols = [
    "drive_status", "drive_file_id", "drive_link", "drive_filename",
    "download_bytes", "sha256", "drive_reason", "dedupe_method", "dedupe_confidence"
]
notion_cols = ["notion_status", "notion_page_id", "notion_page_url", "notion_reason"]

attempts = drive_df.merge(
    notion_df[["paper_id"] + [c for c in notion_cols if c in notion_df.columns]],
    on="paper_id",
    how="outer",
    suffixes=("", "_notion"),
)

# Add domains for quick diagnostics
attempts["chosen_domain"] = attempts.get("chosen_url", "").apply(_domain)
attempts["landing_domain"] = attempts.get("landing_url", "").apply(_domain)

# -----------------------------
# Core metrics
# -----------------------------
total_targets = int(os.getenv("TOTAL_TARGETS_OVERRIDE", "0")) or None
if total_targets is None:
    # best-effort guess: count of unique paper_ids across all logs
    total_targets = attempts["paper_id"].nunique()

fetch_ok = int(drive_df["paper_id"].nunique())
drive_uploaded = int((drive_df.get("drive_status", "") == "uploaded").sum()) if "drive_status" in drive_df.columns else 0
drive_dup_skipped = int((drive_df.get("drive_status", "") == "skipped_duplicate").sum()) if "drive_status" in drive_df.columns else 0

notion_created = int((notion_df.get("notion_status", "") == "created").sum()) if "notion_status" in notion_df.columns else 0
notion_updated = int((notion_df.get("notion_status", "") == "updated").sum()) if "notion_status" in notion_df.columns else 0
notion_ok = notion_created + notion_updated

fetch_fail = int((fail_df.get("stage", "") == "fetch").sum()) if "stage" in fail_df.columns else 0
drive_fail = int((fail_df.get("stage", "") == "drive").sum()) if "stage" in fail_df.columns else 0
notion_fail = int((fail_df.get("stage", "") == "notion").sum()) if "stage" in fail_df.columns else 0

def _rate(num: int, den: int) -> float:
    return 0.0 if den <= 0 else round(num / den, 4)

summary = pd.DataFrame([{
    "targets_total": total_targets,
    "fetch_ok": fetch_ok,
    "fetch_fail": fetch_fail,
    "fetch_ok_rate": _rate(fetch_ok, total_targets),

    "drive_uploaded": drive_uploaded,
    "drive_skipped_duplicate": drive_dup_skipped,
    "drive_fail": drive_fail,
    "drive_upload_rate_per_target": _rate(drive_uploaded, total_targets),
    "drive_upload_rate_per_fetch_ok": _rate(drive_uploaded, max(fetch_ok, 1)),

    "notion_ok": notion_ok,
    "notion_created": notion_created,
    "notion_updated": notion_updated,
    "notion_fail": notion_fail,
    "notion_ok_rate_per_target": _rate(notion_ok, total_targets),
    "notion_ok_rate_per_fetch_ok": _rate(notion_ok, max(fetch_ok, 1)),
}])

print("\n=== Run Summary ===")
display(summary)

# -----------------------------
# Breakdown tables
# -----------------------------
print("\n=== Drive status breakdown ===")
display(_value_counts(drive_df, "drive_status", topn=20))

print("\n=== Notion status breakdown ===")
display(_value_counts(notion_df, "notion_status", topn=20))

print("\n=== Failure stages breakdown ===")
display(_value_counts(fail_df, "stage", topn=20))

print("\n=== Top failure reasons (by stage) ===")
if {"stage", "error_message"}.issubset(set(fail_df.columns)):
    fail_df["error_message_short"] = fail_df["error_message"].fillna("").astype(str).str.slice(0, 120)
    top_reasons = (
        fail_df.groupby(["stage", "error_message_short"])
        .size()
        .reset_index(name="count")
        .sort_values(["stage", "count"], ascending=[True, False])
        .groupby("stage")
        .head(15)
        .reset_index(drop=True)
    )
    display(top_reasons)
else:
    display(pd.DataFrame({"note": ["failures.csv missing expected columns (stage, error_message)."]}))

print("\n=== Publisher/domain hints (chosen_url domain) ===")
if "chosen_domain" in attempts.columns:
    dom = attempts["chosen_domain"].fillna("").astype(str)
    dom_counts = dom.value_counts().head(25).reset_index()
    dom_counts.columns = ["chosen_domain", "count"]
    display(dom_counts)

# Fetch failures by attempted domain (best-effort from attempts_json)
print("\n=== Fetch failures: inferred domains from attempts_json ===")
if not fail_df.empty and "attempts_json" in fail_df.columns and "stage" in fail_df.columns:
    f = fail_df[fail_df["stage"] == "fetch"].copy()
    if not f.empty:
        def infer_domains_from_attempts_json(s: str) -> List[str]:
            s = str(s or "")
            try:
                arr = json.loads(s)
                out = []
                for a in arr:
                    u = str(a.get("url", "")).strip()
                    d = _domain(u)
                    if d:
                        out.append(d)
                return out
            except Exception:
                return []

        f["attempt_domains"] = f["attempts_json"].apply(infer_domains_from_attempts_json)
        exploded = f.explode("attempt_domains")
        dom_fail = (
            exploded["attempt_domains"].fillna("").astype(str).value_counts().head(30).reset_index()
        )
        dom_fail.columns = ["attempt_domain", "fetch_fail_count"]
        display(dom_fail)
    else:
        display(pd.DataFrame({"note": ["No fetch failures found."]}))
else:
    display(pd.DataFrame({"note": ["failures.csv missing attempts_json/stage."]}))

# -----------------------------
# Optional export
# -----------------------------
if EXPORT_METRICS:
    summary_path = METRICS_DIR / "summary.csv"
    attempts_path = METRICS_DIR / "attempts_joined.csv"
    top_reasons_path = METRICS_DIR / "top_failure_reasons.csv"

    summary.to_csv(summary_path, index=False)
    attempts.to_csv(attempts_path, index=False)

    if "top_reasons" in locals():
        top_reasons.to_csv(top_reasons_path, index=False)

    print("\n✅ Exported metrics to:", METRICS_DIR)
    print(" -", summary_path)
    print(" -", attempts_path)
    if "top_reasons" in locals():
        print(" -", top_reasons_path)

# Done



=== Metrics Inputs ===
RUN_DIR: artifacts/019_runs/20260112T044552Z
drive_results: artifacts/019_runs/20260112T044552Z/drive_results.csv
notion_results: artifacts/019_runs/20260112T044552Z/notion_results.csv
failures: artifacts/019_runs/20260112T044552Z/failures.csv
EXPORT_METRICS: True

✅ Loaded:
 - drive_df: (17, 18)
 - notion_df: (17, 8)
 - fail_df : (33, 7)

=== Run Summary ===


,targets_total,fetch_ok,fetch_fail,fetch_ok_rate,drive_uploaded,drive_skipped_duplicate,drive_fail,drive_upload_rate_per_target,drive_upload_rate_per_fetch_ok,notion_ok,notion_created,notion_updated,notion_fail,notion_ok_rate_per_target,notion_ok_rate_per_fetch_ok
0,17,17,33,1.0,16,1,0,0.9412,0.9412,17,17,0,0,1.0,1.0



=== Drive status breakdown ===


,count,count
0,uploaded,16
1,skipped_duplicate,1



=== Notion status breakdown ===


...


=== Publisher/domain hints (chosen_url domain) ===


,chosen_domain,count
0,link.springer.com,5
1,arxiv.org,5
2,www.frontiersin.org,2
3,www.ijfmr.com,2
4,www.nature.com,1
5,wjarr.com,1
6,imafungus.pensoft.net,1



=== Fetch failures: inferred domains from attempts_json ===


,attempt_domain,fetch_fail_count
0,doi.org,33
1,dx.doi.org,33
2,onlinelibrary.wiley.com,16
3,www.mdpi.com,8
4,www.tandfonline.com,3
5,academic.oup.com,2
6,www.annualreviews.org,1
7,osf.io,1
8,iopscience.iop.org,1
9,www.pnas.org,1



✅ Exported metrics to: artifacts/019_runs/20260112T044552Z/metrics
 - artifacts/019_runs/20260112T044552Z/metrics/summary.csv
 - artifacts/019_runs/20260112T044552Z/metrics/attempts_joined.csv
 - artifacts/019_runs/20260112T044552Z/metrics/top_failure_reasons.csv


In [33]:
# ============================================================
# Cell 13: Persist results and logs for downstream use
# ============================================================
#
# This cell consolidates all run artifacts (raw + metrics) and writes:
# - A single "run_manifest.json" describing what happened and where outputs live
# - A "run_summary.md" for human-friendly review
# - Copies (or symlinks) key outputs into a stable "latest" directory
# - Optional: gzip large CSVs to save space
#
# Expected inputs created earlier:
# - RUN_DIR (Path)
# - drive_results.csv, notion_results.csv, failures.csv (Cell 11)
# - metrics/summary.csv, metrics/attempts_joined.csv, metrics/top_failure_reasons.csv (Cell 12, optional)
#

from __future__ import annotations

import os
import json
import gzip
import shutil
import pathlib
from datetime import datetime
from typing import Dict, Any, List

import pandas as pd

RUN_DIR = pathlib.Path(RUN_DIR)
assert RUN_DIR.exists(), f"RUN_DIR not found: {RUN_DIR}"

# -----------------------------
# Config
# -----------------------------
LATEST_DIR = pathlib.Path(os.getenv("LATEST_DIR", "./artifacts/019_runs/latest"))
LATEST_DIR.mkdir(parents=True, exist_ok=True)

# If true, copy files into LATEST_DIR (safe). If false, create symlinks (faster).
USE_COPY_FOR_LATEST = os.getenv("USE_COPY_FOR_LATEST", "true").lower() in ("1", "true", "yes")

# If true, gzip large CSVs (keeps original too unless you set DELETE_ORIGINAL_AFTER_GZIP)
GZIP_LARGE_CSV = os.getenv("GZIP_LARGE_CSV", "true").lower() in ("1", "true", "yes")
GZIP_MIN_BYTES = int(os.getenv("GZIP_MIN_BYTES", str(5 * 1024 * 1024)))  # 5MB
DELETE_ORIGINAL_AFTER_GZIP = os.getenv("DELETE_ORIGINAL_AFTER_GZIP", "false").lower() in ("1", "true", "yes")

# Optional additional metadata
PIPELINE_NAME = os.getenv("PIPELINE_NAME", "019_pdf_fetch_and_ingest_to_drive_notion")
RUN_TS = os.getenv("RUN_TS", RUN_DIR.name)

print("=== Persist Config ===")
print("RUN_DIR:", RUN_DIR)
print("LATEST_DIR:", LATEST_DIR)
print("USE_COPY_FOR_LATEST:", USE_COPY_FOR_LATEST)
print("GZIP_LARGE_CSV:", GZIP_LARGE_CSV, "| min_bytes:", GZIP_MIN_BYTES, "| delete_original:", DELETE_ORIGINAL_AFTER_GZIP)
print("RUN_TS:", RUN_TS)
print("======================\n")

# -----------------------------
# Locate expected files
# -----------------------------
def p(*parts) -> pathlib.Path:
    return RUN_DIR.joinpath(*parts)

drive_csv = p("drive_results.csv")
notion_csv = p("notion_results.csv")
failures_csv = p("failures.csv")
metrics_dir = p("metrics")

paths = {
    "drive_results_csv": drive_csv if drive_csv.exists() else None,
    "notion_results_csv": notion_csv if notion_csv.exists() else None,
    "failures_csv": failures_csv if failures_csv.exists() else None,
    "metrics_summary_csv": (metrics_dir / "summary.csv") if (metrics_dir / "summary.csv").exists() else None,
    "metrics_attempts_joined_csv": (metrics_dir / "attempts_joined.csv") if (metrics_dir / "attempts_joined.csv").exists() else None,
    "metrics_top_failure_reasons_csv": (metrics_dir / "top_failure_reasons.csv") if (metrics_dir / "top_failure_reasons.csv").exists() else None,
}

missing = [k for k, v in paths.items() if v is None and k.endswith("_csv")]
print("Detected files:")
for k, v in paths.items():
    print(f"- {k}: {v}")
if missing:
    print("\n⚠️ Missing some expected CSVs:", missing)

# -----------------------------
# Helpers
# -----------------------------
def file_info(path: pathlib.Path) -> Dict[str, Any]:
    st = path.stat()
    return {
        "path": str(path),
        "bytes": int(st.st_size),
        "modified_utc": datetime.utcfromtimestamp(st.st_mtime).isoformat() + "Z",
    }

def gzip_if_large(path: pathlib.Path) -> pathlib.Path | None:
    if not path.exists():
        return None
    if path.suffix.lower() != ".csv":
        return None
    if path.stat().st_size < GZIP_MIN_BYTES:
        return None

    gz_path = path.with_suffix(path.suffix + ".gz")
    with open(path, "rb") as f_in, gzip.open(gz_path, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

    if DELETE_ORIGINAL_AFTER_GZIP:
        path.unlink()

    return gz_path

def safe_link_or_copy(src: pathlib.Path, dst: pathlib.Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()

    if USE_COPY_FOR_LATEST:
        shutil.copy2(src, dst)
    else:
        # symlink for speed
        dst.symlink_to(src.resolve())

# -----------------------------
# Optional compression
# -----------------------------
compressed: Dict[str, str] = {}
if GZIP_LARGE_CSV:
    for k, path in list(paths.items()):
        if not path:
            continue
        gz = gzip_if_large(path)
        if gz:
            compressed[k + "_gz"] = str(gz)
            print(f"🗜️ gzipped: {path.name} -> {gz.name}")

# -----------------------------
# Human-friendly summary Markdown
# -----------------------------
summary_md = RUN_DIR / "run_summary.md"

def _maybe_read_csv(path: pathlib.Path) -> pd.DataFrame | None:
    if path and path.exists():
        try:
            return pd.read_csv(path)
        except Exception:
            return None
    return None

drive_df = _maybe_read_csv(paths["drive_results_csv"]) if paths["drive_results_csv"] else None
notion_df = _maybe_read_csv(paths["notion_results_csv"]) if paths["notion_results_csv"] else None
fail_df = _maybe_read_csv(paths["failures_csv"]) if paths["failures_csv"] else None
metrics_summary_df = _maybe_read_csv(paths["metrics_summary_csv"]) if paths["metrics_summary_csv"] else None

def _count(df: pd.DataFrame | None) -> int:
    return 0 if df is None else int(len(df))

def _vc(df: pd.DataFrame | None, col: str, topn: int = 8) -> List[str]:
    if df is None or col not in df.columns:
        return []
    vc = df[col].fillna("").astype(str).value_counts().head(topn)
    return [f"{idx}: {int(cnt)}" for idx, cnt in vc.items()]

lines = []
lines.append(f"# Run Summary: {PIPELINE_NAME}")
lines.append("")
lines.append(f"- Run ID: `{RUN_TS}`")
lines.append(f"- Run dir: `{RUN_DIR}`")
lines.append("")
lines.append("## Outputs")
for k, path in paths.items():
    if path and pathlib.Path(path).exists():
        lines.append(f"- {k}: `{path}`")
for k, path in compressed.items():
    lines.append(f"- {k}: `{path}`")

lines.append("")
lines.append("## Counts")
lines.append(f"- Drive rows: {_count(drive_df)}")
lines.append(f"- Notion rows: {_count(notion_df)}")
lines.append(f"- Failures rows: {_count(fail_df)}")

if metrics_summary_df is not None and not metrics_summary_df.empty:
    lines.append("")
    lines.append("## Metrics (from Cell 12)")
    # Write key metrics if present
    row0 = metrics_summary_df.iloc[0].to_dict()
    for key in [
        "targets_total",
        "fetch_ok", "fetch_fail", "fetch_ok_rate",
        "drive_uploaded", "drive_skipped_duplicate", "drive_fail",
        "notion_ok", "notion_created", "notion_updated", "notion_fail",
    ]:
        if key in row0:
            lines.append(f"- {key}: {row0[key]}")

lines.append("")
lines.append("## Drive status breakdown (top)")
for s in _vc(drive_df, "drive_status"):
    lines.append(f"- {s}")

lines.append("")
lines.append("## Notion status breakdown (top)")
for s in _vc(notion_df, "notion_status"):
    lines.append(f"- {s}")

lines.append("")
lines.append("## Failure stages (top)")
for s in _vc(fail_df, "stage"):
    lines.append(f"- {s}")

summary_md.write_text("\n".join(lines), encoding="utf-8")
print("✅ Wrote:", summary_md)

# -----------------------------
# Manifest JSON (machine-friendly)
# -----------------------------
manifest_path = RUN_DIR / "run_manifest.json"

manifest: Dict[str, Any] = {
    "pipeline": PIPELINE_NAME,
    "run_id": RUN_TS,
    "run_dir": str(RUN_DIR),
    "created_utc": datetime.utcnow().isoformat() + "Z",
    "files": {},
    "compressed": compressed,
    "notes": {
        "latest_dir": str(LATEST_DIR),
        "use_copy_for_latest": USE_COPY_FOR_LATEST,
        "gzip_large_csv": GZIP_LARGE_CSV,
        "gzip_min_bytes": GZIP_MIN_BYTES,
        "delete_original_after_gzip": DELETE_ORIGINAL_AFTER_GZIP,
    },
}

for k, path in paths.items():
    if path and pathlib.Path(path).exists():
        manifest["files"][k] = file_info(path)

# include summary files
manifest["files"]["run_summary_md"] = file_info(summary_md)
manifest["files"]["run_manifest_json"] = {"path": str(manifest_path)}  # self reference

manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("✅ Wrote:", manifest_path)

# -----------------------------
# Update stable "latest" directory
# -----------------------------
# We store the key artifacts under latest/ with fixed names,
# so downstream notebooks can always read the newest outputs.
latest_map = {
    "drive_results.csv": drive_csv,
    "notion_results.csv": notion_csv,
    "failures.csv": failures_csv,
    "run_summary.md": summary_md,
    "run_manifest.json": manifest_path,
}

# Also include metrics if present
if paths["metrics_summary_csv"]:
    latest_map["metrics_summary.csv"] = paths["metrics_summary_csv"]
if paths["metrics_attempts_joined_csv"]:
    latest_map["metrics_attempts_joined.csv"] = paths["metrics_attempts_joined_csv"]
if paths["metrics_top_failure_reasons_csv"]:
    latest_map["metrics_top_failure_reasons.csv"] = paths["metrics_top_failure_reasons_csv"]

for name, src in latest_map.items():
    if src and pathlib.Path(src).exists():
        dst = LATEST_DIR / name
        safe_link_or_copy(src, dst)

print("\n✅ Updated latest outputs in:", LATEST_DIR)

# Optional: also write a pointer file with the latest run id
(LATEST_DIR / "LATEST_RUN_ID.txt").write_text(str(RUN_TS), encoding="utf-8")
print("✅ Wrote:", LATEST_DIR / "LATEST_RUN_ID.txt")

# Done


=== Persist Config ===
RUN_DIR: artifacts/019_runs/20260112T044552Z
LATEST_DIR: artifacts/019_runs/latest
USE_COPY_FOR_LATEST: True
GZIP_LARGE_CSV: True | min_bytes: 5242880 | delete_original: False
RUN_TS: 20260112T044552Z

Detected files:
- drive_results_csv: artifacts/019_runs/20260112T044552Z/drive_results.csv
- notion_results_csv: artifacts/019_runs/20260112T044552Z/notion_results.csv
- failures_csv: artifacts/019_runs/20260112T044552Z/failures.csv
- metrics_summary_csv: artifacts/019_runs/20260112T044552Z/metrics/summary.csv
- metrics_attempts_joined_csv: artifacts/019_runs/20260112T044552Z/metrics/attempts_joined.csv
- metrics_top_failure_reasons_csv: artifacts/019_runs/20260112T044552Z/metrics/top_failure_reasons.csv
✅ Wrote: artifacts/019_runs/20260112T044552Z/run_summary.md
✅ Wrote: artifacts/019_runs/20260112T044552Z/run_manifest.json

✅ Updated latest outputs in: artifacts/019_runs/latest
✅ Wrote: artifacts/019_runs/latest/LATEST_RUN_ID.txt


In [34]:
# ============================================================
# Cell 14: Human-review summary (high-priority failures)
# ============================================================
#
# Goal:
# - Produce a short, human-friendly checklist of what to review after a run.
# - Focus on high-priority items (P1/P2, HIGH/MEDIUM RQ relevance) that failed at:
#   - fetch (could not download PDF)
#   - notion (could not create/update Notion page)
# - Provide "actionable" hints (domain-level patterns, likely paywall, retry candidates).
#
# Inputs:
# - targets_df (selected targets from Cell 04)
# - failures.csv (from Cell 11)
# - drive_results.csv, notion_results.csv (optional for context)
#
# Outputs:
# - RUN_DIR/human_review.md
# - A preview table in the notebook
#

from __future__ import annotations

import os
import re
import json
import pathlib
from urllib.parse import urlparse

import pandas as pd

RUN_DIR = pathlib.Path(RUN_DIR)
failures_csv = RUN_DIR / "failures.csv"
drive_csv = RUN_DIR / "drive_results.csv"
notion_csv = RUN_DIR / "notion_results.csv"

assert failures_csv.exists(), f"Missing: {failures_csv}"
fail_df = pd.read_csv(failures_csv)

drive_df = pd.read_csv(drive_csv) if drive_csv.exists() else pd.DataFrame()
notion_df = pd.read_csv(notion_csv) if notion_csv.exists() else pd.DataFrame()

# targets_df is expected from Cell 04 (selected targets)
if "targets_df" not in globals():
    raise ValueError("targets_df not found. Run Cell 04 first (Load candidates + target selection).")

# -----------------------------
# Review policy (tunable)
# -----------------------------
REVIEW_PRIORITY_TIERS = [x.strip() for x in os.getenv("REVIEW_PRIORITY_TIERS", "P1,P2").split(",") if x.strip()]
REVIEW_RQ_LABELS = [x.strip() for x in os.getenv("REVIEW_RQ_LABELS", "HIGH,MEDIUM").split(",") if x.strip()]
REVIEW_STAGES = [x.strip() for x in os.getenv("REVIEW_STAGES", "fetch,notion").split(",") if x.strip()]
MAX_ITEMS_PER_STAGE = int(os.getenv("MAX_ITEMS_PER_STAGE", "30"))

REPORT_PATH = RUN_DIR / "human_review.md"

print("=== Human Review Config ===")
print("RUN_DIR:", RUN_DIR)
print("REVIEW_PRIORITY_TIERS:", REVIEW_PRIORITY_TIERS)
print("REVIEW_RQ_LABELS:", REVIEW_RQ_LABELS)
print("REVIEW_STAGES:", REVIEW_STAGES)
print("MAX_ITEMS_PER_STAGE:", MAX_ITEMS_PER_STAGE)
print("REPORT_PATH:", REPORT_PATH)
print("===========================\n")

# -----------------------------
# Helpers
# -----------------------------
def _domain(url: str) -> str:
    url = str(url or "").strip()
    if not url:
        return ""
    try:
        return urlparse(url).netloc.lower()
    except Exception:
        return ""

def _short(s: str, n: int = 140) -> str:
    s = "" if pd.isna(s) else str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s[:n] + ("…" if len(s) > n else "")

def _as_list(s):
    if pd.isna(s) or s is None:
        return []
    if isinstance(s, list):
        return s
    try:
        v = json.loads(str(s))
        return v if isinstance(v, list) else []
    except Exception:
        return []

# -----------------------------
# Build "priority universe" from targets_df
# -----------------------------
t = targets_df.copy()

# Best-effort: normalize typical columns coming from 018/019
for col in ["priority_tier", "rq_relevance_label", "free_pdf_label", "paper_id", "title", "year", "doi", "landing_url"]:
    if col not in t.columns:
        t[col] = None

# Filter to high priority targets
priority_targets = t[
    (t["priority_tier"].astype(str).isin(REVIEW_PRIORITY_TIERS))
    & (t["rq_relevance_label"].astype(str).isin(REVIEW_RQ_LABELS))
].copy()

priority_targets["landing_domain"] = priority_targets["landing_url"].apply(_domain)

print(f"✅ Priority targets: {len(priority_targets)} / {len(t)}")

# -----------------------------
# Join failures with priority targets
# -----------------------------
f = fail_df.copy()
for col in ["stage", "paper_id", "title", "error_code", "error_message", "attempts_json"]:
    if col not in f.columns:
        f[col] = None

f = f[f["stage"].astype(str).isin(REVIEW_STAGES)].copy()

review = priority_targets.merge(
    f,
    on=["paper_id", "title"],
    how="inner",
    suffixes=("", "_fail"),
)

# Add a domain hint from attempts_json (if present)
def infer_attempt_domains(attempts_json: str) -> str:
    arr = _as_list(attempts_json)
    ds = []
    for a in arr:
        try:
            u = str(a.get("url", "")).strip()
            d = _domain(u)
            if d:
                ds.append(d)
        except Exception:
            pass
    ds = list(dict.fromkeys(ds))  # unique preserving order
    return ", ".join(ds[:4])

review["attempt_domains"] = review["attempts_json"].apply(infer_attempt_domains)
review["error_message_short"] = review["error_message"].apply(lambda x: _short(x, 160))

# -----------------------------
# Stage-specific views
# -----------------------------
review_fetch = review[review["stage"] == "fetch"].copy()
review_notion = review[review["stage"] == "notion"].copy()

# Sort: higher priority_score first if available
if "priority_score" in review_fetch.columns:
    review_fetch = review_fetch.sort_values(["priority_score"], ascending=False)
if "priority_score" in review_notion.columns:
    review_notion = review_notion.sort_values(["priority_score"], ascending=False)

review_fetch = review_fetch.head(MAX_ITEMS_PER_STAGE)
review_notion = review_notion.head(MAX_ITEMS_PER_STAGE)

# -----------------------------
# Add "context" links if available
# -----------------------------
drive_link_map = {}
if not drive_df.empty and "paper_id" in drive_df.columns and "drive_link" in drive_df.columns:
    drive_link_map = dict(zip(drive_df["paper_id"].astype(str), drive_df["drive_link"].astype(str)))

notion_url_map = {}
if not notion_df.empty and "paper_id" in notion_df.columns and "notion_page_url" in notion_df.columns:
    notion_url_map = dict(zip(notion_df["paper_id"].astype(str), notion_df["notion_page_url"].astype(str)))

def _context_links(paper_id: str) -> dict:
    return {
        "drive_link": drive_link_map.get(str(paper_id), ""),
        "notion_page_url": notion_url_map.get(str(paper_id), ""),
    }

review_fetch["drive_link"] = review_fetch["paper_id"].apply(lambda x: _context_links(x)["drive_link"])
review_fetch["notion_page_url"] = review_fetch["paper_id"].apply(lambda x: _context_links(x)["notion_page_url"])

review_notion["drive_link"] = review_notion["paper_id"].apply(lambda x: _context_links(x)["drive_link"])
review_notion["notion_page_url"] = review_notion["paper_id"].apply(lambda x: _context_links(x)["notion_page_url"])

# -----------------------------
# Quick heuristics for action hints
# -----------------------------
def classify_fetch_issue(msg: str) -> str:
    s = (msg or "").lower()
    if "403" in s or "forbidden" in s:
        return "Likely paywall / bot-block (403). Try OA mirror, author copy, or manual download."
    if "404" in s or "not found" in s:
        return "Not found (404). Candidate URL may be wrong; try alternative discovery."
    if "too small" in s or "empty" in s:
        return "Downloaded content is not a PDF (HTML/landing). Try different PDF URL."
    if "timeout" in s:
        return "Timeout. Retry with longer timeout / backoff."
    return "Needs manual inspection."

review_fetch["action_hint"] = review_fetch["error_message"].apply(classify_fetch_issue)
review_notion["action_hint"] = "Notion mapping/schema issue. Check property names and self-healing map; re-run Cell 10."

# -----------------------------
# Markdown report
# -----------------------------
lines = []
lines.append("# Human Review Checklist (High-Priority Failures)")
lines.append("")
lines.append(f"- Run dir: `{RUN_DIR}`")
lines.append(f"- Focus: tiers={REVIEW_PRIORITY_TIERS}, rq={REVIEW_RQ_LABELS}, stages={REVIEW_STAGES}")
lines.append("")

# Domain-level patterns
lines.append("## Domain patterns (fetch failures)")
if not review_fetch.empty:
    dom_counts = review_fetch["attempt_domains"].fillna("").astype(str).value_counts().head(20)
    for dom, cnt in dom_counts.items():
        if dom.strip():
            lines.append(f"- {dom}: {int(cnt)}")
else:
    lines.append("- (none)")

lines.append("")
lines.append("## Fetch failures to review")
if review_fetch.empty:
    lines.append("- (none)")
else:
    for _, r in review_fetch.iterrows():
        lines.append(f"### {r['paper_id']}")
        lines.append(f"- Title: {r['title']}")
        lines.append(f"- Priority: {r.get('priority_tier','')} | RQ: {r.get('rq_relevance_label','')} | Year: {r.get('year','')}")
        if str(r.get("doi","")).strip():
            lines.append(f"- DOI: {r.get('doi','')}")
        if str(r.get("landing_url","")).strip():
            lines.append(f"- Landing URL: {r.get('landing_url','')}")
        if str(r.get("attempt_domains","")).strip():
            lines.append(f"- Attempt domains: {r.get('attempt_domains','')}")
        lines.append(f"- Error: {r.get('error_code','')} | {r.get('error_message_short','')}")
        lines.append(f"- Hint: {r.get('action_hint','')}")
        if str(r.get("drive_link","")).strip():
            lines.append(f"- Drive link: {r.get('drive_link','')}")
        if str(r.get("notion_page_url","")).strip():
            lines.append(f"- Notion page: {r.get('notion_page_url','')}")
        lines.append("")

lines.append("## Notion failures to review")
if review_notion.empty:
    lines.append("- (none)")
else:
    for _, r in review_notion.iterrows():
        lines.append(f"### {r['paper_id']}")
        lines.append(f"- Title: {r['title']}")
        lines.append(f"- Priority: {r.get('priority_tier','')} | RQ: {r.get('rq_relevance_label','')} | Year: {r.get('year','')}")
        if str(r.get("doi","")).strip():
            lines.append(f"- DOI: {r.get('doi','')}")
        if str(r.get("landing_url","")).strip():
            lines.append(f"- Landing URL: {r.get('landing_url','')}")
        lines.append(f"- Error: {r.get('error_code','')} | {r.get('error_message_short','')}")
        lines.append(f"- Hint: {r.get('action_hint','')}")
        if str(r.get("drive_link","")).strip():
            lines.append(f"- Drive link: {r.get('drive_link','')}")
        if str(r.get("notion_page_url","")).strip():
            lines.append(f"- Notion page: {r.get('notion_page_url','')}")
        lines.append("")

REPORT_PATH.write_text("\n".join(lines), encoding="utf-8")
print("✅ Wrote:", REPORT_PATH)

# -----------------------------
# Notebook preview (compact)
# -----------------------------
preview_cols = [
    "stage", "paper_id", "title",
    "priority_tier", "rq_relevance_label", "year",
    "doi", "landing_url",
    "error_code", "error_message_short",
    "attempt_domains", "action_hint",
]
preview_cols = [c for c in preview_cols if c in review.columns]

print("\n=== Preview: High-priority failures (top) ===")
display(review[preview_cols].head(30))


=== Human Review Config ===
RUN_DIR: artifacts/019_runs/20260112T044552Z
REVIEW_PRIORITY_TIERS: ['P1', 'P2']
REVIEW_RQ_LABELS: ['HIGH', 'MEDIUM']
REVIEW_STAGES: ['fetch', 'notion']
MAX_ITEMS_PER_STAGE: 30
REPORT_PATH: artifacts/019_runs/20260112T044552Z/human_review.md

✅ Priority targets: 50 / 50
✅ Wrote: artifacts/019_runs/20260112T044552Z/human_review.md

=== Preview: High-priority failures (top) ===


,stage,paper_id,title,priority_tier,rq_relevance_label,year,doi,landing_url,error_code,error_message_short,attempt_domains
0,fetch,doi:10.1002/sej.1515,Venture capital exit after venture IPO,P1,HIGH,2024,10.1002/sej.1515,https://doi.org/10.1002/sej.1515,paywall_suspected,http_status_403,"onlinelibrary.wiley.com, doi.org, dx.doi.org"
1,fetch,doi:10.1111/1467-8551.12803,Post‐IPO lead venture capital firm involvement...,P1,HIGH,2024,10.1111/1467-8551.12803,https://doi.org/10.1111/1467-8551.12803,paywall_suspected,http_status_403,"onlinelibrary.wiley.com, doi.org, dx.doi.org"
2,fetch,doi:10.1111/jfir.12412,"VC ownership post‐IPO: When, why, and how do V...",P1,HIGH,2024,10.1111/jfir.12412,https://doi.org/10.1111/jfir.12412,paywall_suspected,http_status_403,"onlinelibrary.wiley.com, doi.org, dx.doi.org"
3,fetch,doi:10.1146/annurev-financial-111021-100657,IPOs and SPACs: Recent Developments,P2,MEDIUM,2023,10.1146/annurev-financial-111021-100657,https://doi.org/10.1146/annurev-financial-1110...,paywall_suspected,http_status_403,"www.annualreviews.org, doi.org, dx.doi.org"
4,fetch,doi:10.1093/rfs/hhad071,Common Venture Capital Investors and Startup G...,P2,MEDIUM,2023,10.1093/rfs/hhad071,https://doi.org/10.1093/rfs/hhad071,paywall_suspected,http_status_403,"academic.oup.com, doi.org, dx.doi.org"
5,fetch,doi:10.3390/systems12030072,Overcoming Uncertainty in Novel Technologies: ...,P2,HIGH,2024,10.3390/systems12030072,https://doi.org/10.3390/systems12030072,paywall_suspected,http_status_403,"www.mdpi.com, doi.org, dx.doi.org"
6,fetch,doi:10.3390/app151810060,Prioritizing Early-Stage Start-Up Investment A...,P2,HIGH,2025,10.3390/app151810060,https://doi.org/10.3390/app151810060,paywall_suspected,http_status_403,"www.mdpi.com, doi.org, dx.doi.org"
7,fetch,doi:10.3390/jrfm17040159,Pathways to Success: The Interplay of Industry...,P2,HIGH,2024,10.3390/jrfm17040159,https://doi.org/10.3390/jrfm17040159,paywall_suspected,http_status_403,"www.mdpi.com, doi.org, dx.doi.org"
8,fetch,doi:10.1002/bse.3587,Is biodiversity disclosure emerging as a key t...,P2,MEDIUM,2023,10.1002/bse.3587,https://doi.org/10.1002/bse.3587,paywall_suspected,http_status_403,"onlinelibrary.wiley.com, doi.org, dx.doi.org"
9,fetch,doi:10.1002/anie.202425439,g‐C<sub>3</sub>N<sub>4</sub> S‐Scheme Homojunc...,P2,MEDIUM,2025,10.1002/anie.202425439,https://doi.org/10.1002/anie.202425439,paywall_suspected,http_status_403,"onlinelibrary.wiley.com, doi.org, dx.doi.org"
